# BirdCLEF+2026: HGNetV2-B0 Baseline [Training]

【UPDATE1】 I added AttnetionSEDHead and CustomLoss for it.  
【UPDATE2】 I renewed CustomLoss, which adopts logit-LSE.  
【UPDATE3】 I replaced the classifier head with LSEHead and used BCEWithLogitsLoss.

## About

This notebook shows you an example of training process.

I used some techniques for fast audio loading:  
- converted `.ogg` files into `.wav` in advance
    - https://www.kaggle.com/datasets/ttahara/birdclef2026-train-audio-wav-00
    - https://www.kaggle.com/datasets/ttahara/birdclef2026-train-audio-wav-01
    - https://www.kaggle.com/datasets/ttahara/birdclef2026-train-audio-wav-02
    - https://www.kaggle.com/datasets/ttahara/birdclef2026-train-audio-wav-03
- load **not** entire the `.wav`file but only **the necessary parts(5 sec)** of it using `soundfile` library

I trained `HGNetV2-B0` by 4-fold cross validation.  
Each fold took a little less than 30 minutes, therefore the entire training process completed in less than 2 hours.

### settings
#### data
* input : `train_audio` and `train_soundscapes`
* target: `primary_label` and `secondary_labels`
* CV split:
    * Multi-Label Stratifiled Group K-Fold(K=4)
    * using file ids as group ids
* LogMelSpectrogram:
    ```python
    mel_spectrogram_params = dict(
        sample_rate= 32_000,
        n_fft      = 2048,
        win_length = 626,
        hop_length = 313,
        f_min      = 20,
        n_mels     = 256,
        power      = 2.0,
        center     = True,
        pad_mode   = "reflect",
        norm       = "slaney",
        mel_scale  = 'htk',
    )
    top_db = 80
    lms_shape = (256, 256)
    ```
#### model
* backbone: `hgnetv2_b0.ssld_stage2_ft_in1k` from `timm`
* head    : LSEHead(head_dropout=0.5, temperature=1.0)

#### training
* max_epoch : 20
* batch_size: 64
* optimizer: AdamW(`lr`=5e-4, `weight_decay`=1e-4)
* scheduler: OneCosineLR(`max_lr`=5e-4, `min_lr`=5e-6, `warmup_epoch`=5)
* data augmentation: MixUp(`alpha`=1.0, `theta`=0.8)
* use amp training: True
* loss: `nn.BCEWithLogitsLoss`

> https://www.kaggle.com/competitions/birdclef-2026/writeups/5th-place-solution-both-are-all-you-need

## Preparation

### libraries

In [1]:
!uv pip install openvino onnxscript iterative-stratification onnxruntime-gpu

Using Python 3.12.12 environment at: /usr
Resolved 19 packages in 312ms
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
openvino-telemetry   ------------------------------ 16.00 KiB/24.64 KiB
⠙ Preparing packages... (0/6)
openvino-telemetry   ------------------------------ 16.00 KiB/24.64 KiB
⠙ Preparing packages... (0/6)
openvino-telemetry   ------------------------------ 24.64 KiB/24.64 KiB
⠙ Preparing packages... (0/6)
openvino-telemetry   ------------------------------ 24.64 KiB/24.64 KiB
⠙ Preparing packages... (0/6)
openvino-telemetry   ------------------------------ 24.64 KiB/24.64 KiB
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
⠙ Preparing packages... (0/6)
onnx-ir              ------------------------------     0 B/162.88 KiB
⠙ Preparing packages... (0/6)
onnx-ir              ------------------------------     0 B/162.88 KiB
⠙ Preparing packages... (0/6)
onnx-i

In [2]:
RANDOM_SEED = 1086

import os
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

In [3]:
import ast
import gc
import copy
import math
import random
import typing as tp
from pathlib import Path

from time import time
from functools import wraps
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

from tqdm.notebook import tqdm, trange
from sklearn.metrics import roc_auc_score, log_loss

import wave
import soundfile

import timm
import torch
import torchaudio
from torchvision.transforms import v2 as tvt_v2
import torchaudio.transforms as AT

from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR, LinearLR, SequentialLR
from torch.optim import AdamW

from huggingface_hub import hf_hub_download

import joblib
import openvino as ov
import onnxruntime as ort

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
ROOT = Path.cwd().parent

INPUT = ROOT / "input"
DATA = INPUT / "competitions" / "birdclef-2026"
TRAIN_AUDIO = DATA / "train_audio"
TRAIN_SS = DATA / "train_soundscapes"
TEST_SS = DATA / "test_soundscapes"

TRAIN_AUDIO_WAVS = [
    INPUT / "datasets" / f"ttahara/birdclef2026-train-audio-wav-{i:02}"
    for i in range(4)]
PROC = ROOT / "processed_data"
TRAIN_SS_SPLIT = PROC / "train_soundscapes_split"
TRAIN_SS_SPLIT.mkdir(exist_ok=True, parents=True)

N_FOLDS = 4
N_CLASSES = 234

In [6]:
def set_random_seed(seed: int = 520, deterministic: bool = True):
    """Set seeds"""
    os.environ["PYTHONHASHSEED"] = str(seed)  # python
    random.seed(seed)  # python
    np.random.seed(seed)  # cpu
    torch.manual_seed(seed)  # cpu
    if torch.cuda.is_available():  # gpu
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic

set_random_seed(RANDOM_SEED)

### load label file

In [7]:
train_labels = pd.read_csv(DATA / "train.csv")

train_ss_labels = pd.read_csv(DATA / "train_soundscapes_labels.csv")
train_ss_labels = train_ss_labels.drop_duplicates().reset_index(drop=True)

train_ss_labels["start"] = pd.to_datetime(train_ss_labels["start"], format="%H:%M:%S")
train_ss_labels["end"] = pd.to_datetime(train_ss_labels["end"], format="%H:%M:%S")
train_ss_labels["start_sec"] = train_ss_labels["start"].dt.minute * 60 + train_ss_labels["start"].dt.second
train_ss_labels["end_sec"] = train_ss_labels["end"].dt.minute * 60 + train_ss_labels["end"].dt.second

In [8]:
taxonomy = pd.read_csv(DATA / "taxonomy.csv")

CLASSES = taxonomy.primary_label.values.tolist()

label2idx = {label: idx for idx, label in enumerate(taxonomy.primary_label.values)}
idx2label = {idx: label for label, idx in label2idx.items()}

### split labeled audio files in train_soundscapes

In [9]:
train_ss_ai_list = []
train_ss_fn_list = []
train_ss_pl_list = []
sampling_rate = 32_000

tmp_in_fn, tmp_pl, tmp_start, tmp_end = train_ss_labels.loc[
    0, ["filename", "primary_label", "start_sec", "end_sec"]].values
tmp_wave, _ = soundfile.read(TRAIN_SS / tmp_in_fn, dtype="float32")

for in_fn, pl, start, end in tqdm(
    train_ss_labels.loc[1:, ["filename", "primary_label", "start_sec", "end_sec"]].values
):
    if in_fn == tmp_in_fn and pl == tmp_pl:
        tmp_end = end
        continue
    
    wave_seg = tmp_wave[tmp_start * sampling_rate: tmp_end * sampling_rate]
    ai = tmp_in_fn.split(".")[0] 
    out_fn = f"{ai}_{tmp_start}-{tmp_end}.wav"
    soundfile.write(
        str(TRAIN_SS_SPLIT / out_fn), wave_seg, samplerate=sampling_rate,
        format="wav", subtype="FLOAT")
    
    train_ss_ai_list.append(ai)
    train_ss_fn_list.append(out_fn)
    train_ss_pl_list.append(tmp_pl)

    tmp_pl = pl
    tmp_start = start
    tmp_end = end

    if in_fn != tmp_in_fn:
        tmp_in_fn = in_fn
        tmp_wave, _ = soundfile.read(TRAIN_SS / tmp_in_fn, dtype="float32")

else:
    wave_seg = tmp_wave[tmp_start * sampling_rate: tmp_end * sampling_rate]
    ai = tmp_in_fn.split(".")[0]
    out_fn = f"{ai}_{start}-{end}.wav"
    soundfile.write(
        str(TRAIN_SS_SPLIT / out_fn), wave_seg, samplerate=sampling_rate,
        format="wav", subtype="FLOAT")
    
    train_ss_ai_list.append(ai)
    train_ss_fn_list.append(out_fn)
    train_ss_pl_list.append(tmp_pl)

  0%|          | 0/738 [00:00<?, ?it/s]

### prepare train label dataframe

In [10]:
train_ss_labels_merged = pd.DataFrame({
    "audio_id": train_ss_ai_list,
    "filename": train_ss_fn_list,
    "primary_label": train_ss_pl_list,
    "labels": train_ss_pl_list,
})
train_ss_labels_merged["file_path"] = [
    str(TRAIN_SS_SPLIT / fn) for fn in train_ss_labels_merged["filename"].values]

In [11]:
print(train_ss_labels_merged.shape)

(413, 5)


In [12]:
display(train_ss_labels.head(12))

,filename,start,end,primary_label,start_sec,end_sec
0,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:00,1900-01-01 00:00:05,22961;23158;24321;517063;65380,0,5
1,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:05,1900-01-01 00:00:10,22961;23158;24321;517063;65380,5,10
2,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:10,1900-01-01 00:00:15,22961;23158;24321;517063;65380,10,15
3,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:15,1900-01-01 00:00:20,22961;23158;24321;517063;65380,15,20
4,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:20,1900-01-01 00:00:25,22961;23158;24321;517063;65380,20,25
5,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:25,1900-01-01 00:00:30,22961;23158;24321;517063;65380,25,30
6,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:30,1900-01-01 00:00:35,22961;23158;24321;517063;65380,30,35
7,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:35,1900-01-01 00:00:40,22961;23158;24321;517063;65380,35,40
8,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:40,1900-01-01 00:00:45,22961;23158;24321;65380,40,45
9,BC2026_Train_0039_S22_20211231_201500.ogg,1900-01-01 00:00:45,1900-01-01 00:00:50,22961;23158;24321;517063;65380,45,50


In [13]:
display(train_ss_labels_merged.head(5))

,audio_id,filename,primary_label,labels,file_path
0,BC2026_Train_0039_S22_20211231_201500,BC2026_Train_0039_S22_20211231_201500_0-40.wav,22961;23158;24321;517063;65380,22961;23158;24321;517063;65380,/kaggle/processed_data/train_soundscapes_split...
1,BC2026_Train_0039_S22_20211231_201500,BC2026_Train_0039_S22_20211231_201500_40-45.wav,22961;23158;24321;65380,22961;23158;24321;65380,/kaggle/processed_data/train_soundscapes_split...
2,BC2026_Train_0039_S22_20211231_201500,BC2026_Train_0039_S22_20211231_201500_45-50.wav,22961;23158;24321;517063;65380,22961;23158;24321;517063;65380,/kaggle/processed_data/train_soundscapes_split...
3,BC2026_Train_0039_S22_20211231_201500,BC2026_Train_0039_S22_20211231_201500_50-55.wav,22961;23158;24321;65380,22961;23158;24321;65380,/kaggle/processed_data/train_soundscapes_split...
4,BC2026_Train_0039_S22_20211231_201500,BC2026_Train_0039_S22_20211231_201500_55-60.wav,22961;23158;24321;517063;65380,22961;23158;24321;517063;65380,/kaggle/processed_data/train_soundscapes_split...


In [14]:
pl2wp = {}
for i in range(4):
    wav_path = TRAIN_AUDIO_WAVS[i]
    for pl_dir in sorted(wav_path.iterdir()):
        pl2wp[pl_dir.name] = wav_path

In [15]:
train_labels_merged = pd.DataFrame()
train_labels_merged["audio_id"] = train_labels["filename"].str.split(".").explode().values[0::2]
train_labels_merged["filename"] = train_labels["filename"]

train_labels_merged["primary_label"] = train_labels["primary_label"]
train_labels_merged["labels"] = [
    ";".join([pl] + ast.literal_eval(sls))
    for pl, sls in train_labels[["primary_label", "secondary_labels"]].values
]

train_labels_merged["file_path"] = [
    str(pl2wp[pl] / f"{fn.split(".")[0]}.wav")
    for fn, pl in train_labels_merged[["filename", "primary_label"]].values
]

In [16]:
train_df = pd.concat([
    train_labels_merged,
    train_ss_labels_merged,
], axis=0, ignore_index=True)

In [17]:
labels_arr = np.zeros((len(train_df), len(label2idx)), dtype=np.float32)

for idx, labels in enumerate(train_df["labels"].values):
    for l in labels.split(";"):
        labels_arr[idx, label2idx[l]] = 1

In [18]:
print(train_df.shape)
label_df = pd.DataFrame(
    labels_arr, columns=list(label2idx.keys()))
train_df = pd.concat([train_df, label_df], axis=1)
print(train_df.shape)

(35962, 5)
(35962, 239)


### split folds

In [19]:
class MultiLabelStratifiedGroupKFold:

    def __init__(self, n_splits: int, random_state: int):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, label_arr: np.array, gid_arr: np.array):
        """
        create multi-label stratified group kfold indexs.
    
        reference: https://www.kaggle.com/jakubwasikowski/stratified-group-k-fold-cross-validation
        input:
            label_arr: numpy.ndarray, shape = (n_train, n_class)
                multi-label for each sample's index using multi-hot vectors
            gid_arr: numpy.array, shape = (n_train,)
                group id for each sample's index
        output:
            yield indexs array list for each fold's train and validation.
        """
        np.random.seed(self.random_state)
        random.seed(self.random_state)
        start_time = time()
        n_train, n_class = label_arr.shape
        gid_unique = sorted(set(gid_arr))
        n_group = len(gid_unique)
    
        # # aid_arr: (n_train,), indicates alternative id for group id.
        # # generally, group ids are not 0-index and continuous or not integer.
        gid2aid = dict(zip(gid_unique, range(n_group)))
        # aid2gid = dict(zip(range(n_group), gid_unique))
        aid_arr = np.vectorize(lambda x: gid2aid[x])(gid_arr)
    
        # # count labels by class
        cnts_by_class = label_arr.sum(axis=0)  # (n_class, )
    
        # # count labels by group id.
        col, row = np.array(sorted(enumerate(aid_arr), key=lambda x: x[1])).T
        cnts_by_group = coo_matrix(
            (np.ones(len(label_arr)), (row, col))
        ).dot(coo_matrix(label_arr)).toarray().astype(int)
        del col
        del row
        cnts_by_fold = np.zeros((self.n_splits, n_class), int)
    
        groups_by_fold = [[] for fid in range(self.n_splits)]
        group_and_cnts = list(enumerate(cnts_by_group))  # pair of aid and cnt by group
        np.random.shuffle(group_and_cnts)
        print("finished preparation", time() - start_time)
        for aid, cnt_by_g in sorted(group_and_cnts, key=lambda x: -np.std(x[1])):
            best_fold = None
            min_eval = None
            for fid in range(self.n_splits):
                # # eval assignment.
                cnts_by_fold[fid] += cnt_by_g
                fold_eval = (cnts_by_fold / cnts_by_class).std(axis=0).mean()
                cnts_by_fold[fid] -= cnt_by_g
    
                if min_eval is None or fold_eval < min_eval:
                    min_eval = fold_eval
                    best_fold = fid
    
            cnts_by_fold[best_fold] += cnt_by_g
            groups_by_fold[best_fold].append(aid)
        print("finished assignment.", time() - start_time)
    
        gc.collect()
        idx_arr = np.arange(n_train)
        for fid in range(self.n_splits):
            val_groups = groups_by_fold[fid]
    
            val_indexs_bool = np.isin(aid_arr, val_groups)
            train_indexs = idx_arr[~val_indexs_bool]
            val_indexs = idx_arr[val_indexs_bool]
    
            print("[fold {}]".format(fid), end=" ")
            print("n_group: (train, val) = ({}, {})".format(
                n_group - len(val_groups), len(val_groups)), end=" ")
            print("n_sample: (train, val) = ({}, {})".format(
                len(train_indexs), len(val_indexs)))
    
            yield train_indexs, val_indexs

In [20]:
mlsgkf = MultiLabelStratifiedGroupKFold(n_splits=N_FOLDS, random_state=RANDOM_SEED)
train_val_splits = list(mlsgkf.split(
    labels_arr, train_df["audio_id"].values
))

finished preparation 0.24935269355773926
finished assignment. 6.594892740249634
[fold 0] n_group: (train, val) = (26649, 8966) n_sample: (train, val) = (26901, 9061)
[fold 1] n_group: (train, val) = (26698, 8917) n_sample: (train, val) = (26949, 9013)
[fold 2] n_group: (train, val) = (26727, 8888) n_sample: (train, val) = (26997, 8965)
[fold 3] n_group: (train, val) = (26771, 8844) n_sample: (train, val) = (27039, 8923)


In [21]:
train_df.insert(5, "fold",  -1)
for fold_id, (trn_idx, val_idx) in enumerate(train_val_splits):
    train_df.loc[val_idx, "fold"] = fold_id

In [22]:
for fold_id in range(N_FOLDS):
    print(f"[fold {fold_id}]")
    print(
        "train_audio      :",
        ((train_df["fold"] != fold_id) & train_df['filename'].str.contains("ogg")).sum(),
        ((train_df["fold"] == fold_id) & train_df['filename'].str.contains("ogg")).sum(),
    )
    print(
        "train_soundscapes:",
        ((train_df["fold"] != fold_id) & train_df['filename'].str.contains("wav")).sum(),
        ((train_df["fold"] == fold_id) & train_df['filename'].str.contains("wav")).sum(),
    )

[fold 0]
train_audio      : 26599 8950
train_soundscapes: 302 111
[fold 1]
train_audio      : 26651 8898
train_soundscapes: 298 115
[fold 2]
train_audio      : 26676 8873
train_soundscapes: 321 92
[fold 3]
train_audio      : 26721 8828
train_soundscapes: 318 95


## Training

### definition

#### dataset

In [23]:
class BirdTrainDataset(Dataset):
    def __init__(
        self,
        paths,
        labels,
        is_soundscape,
        sampling_rate=32_000,
        segment_sec=10,  # 実行時に CFG.context_sec (20) が渡されます
    ):
        self.paths = paths
        self.labels = labels
        self.is_soundscape = is_soundscape
        self.sampling_rate = sampling_rate
        self.segment_sec = segment_sec
        self.resample = AT.Resample(self.sampling_rate, int(self.sampling_rate*(1+torch.randint(-5, 5, (1,))*0.01)))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        wave = self._load_audio_file(path, self.is_soundscape[idx])
        label = self.labels[idx].astype(np.float32)
        return {
            "wave": wave,
            "label": label,
        }

    def _load_audio_file(self, path, is_soundscape):
        duration = int(self.sampling_rate * self.segment_sec)

        with soundfile.SoundFile(path) as f:
            n_frames = f.frames

            if n_frames == 0:
                # 破損ファイル等で長さ0の例外処理
                return np.zeros(duration, dtype="float32")

            if n_frames >= duration:
                wave = np.zeros(duration, dtype="float32")
                if is_soundscape:
                    # soundscape split 由来は、できるだけ中央にイベントが来るように中央crop
                    start = max(0, (n_frames - duration) // 2)
                else:
                    # train_audio は従来通り random crop
                    start = np.random.randint(0, n_frames - duration + 1)

                f.seek(start)
                wave[:] = f.read(frames=duration, dtype="float32")
                # wave = self.resample(wave) # <= TBD

            else:
                # ==========================================
                # 【変更点】Wrap Padding (波形の繰り返し)
                # ==========================================
                short_wave = f.read(dtype="float32")
                # short_wave = self.resample(short_wave) # <- TBD
                
                # 必要な長さに到達するまで波形を繰り返す
                repeats = (duration // n_frames) + 1
                wave = np.tile(short_wave, repeats)[:duration]

                if not is_soundscape:
                    # train_audio の場合、繰り返しの「継ぎ目」や開始位置を毎Epochランダムにずらす
                    # これによりモデルが特定の無音パターンを暗記するのを防ぎます
                    roll_shift = np.random.randint(0, n_frames)
                    wave = np.roll(wave, roll_shift)
                else:
                    # soundscapeの場合は中央寄せのニュアンスを残すため、
                    # 元波形の中心が20秒枠の中心付近にくるようにシフト調整
                    shift_amount = (duration - n_frames) // 2
                    wave = np.roll(wave, shift_amount)
                    
        # wave = self.apply_random_gain_db(wave, min_db=CFG.gain_min_db, max_db=CFG.gain_max_db)  # <- TBD

        return wave

    def apply_random_gain_db(self, tensor, min_db=-6, max_db=6):
        gain_db = (max_db - min_db) * torch.rand(1).item() + min_db
        gain_linear = 10 ** (gain_db / 20)
        return tensor * gain_linear


class BirdValidDataset(Dataset):
    def __init__(
        self,
        paths,
        labels,
        is_soundscape,
        sampling_rate=32_000,
        segment_sec=10,
    ):
        self.paths = paths
        self.labels = labels
        self.is_soundscape = is_soundscape
        self.sampling_rate = sampling_rate
        self.segment_sec = segment_sec

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        wave = self._load_audio_file(path, self.is_soundscape[idx])
        label = self.labels[idx].astype(np.float32)
        return {
            "wave": wave,
            "label": label,
        }

    def _load_audio_file(self, path, is_soundscape):
        duration = int(self.sampling_rate * self.segment_sec)

        with soundfile.SoundFile(path) as f:
            n_frames = f.frames

            if n_frames == 0:
                return np.zeros(duration, dtype="float32")

            if n_frames >= duration:
                wave = np.zeros(duration, dtype="float32")
                if is_soundscape:
                    start = max(0, (n_frames - duration) // 2)
                else:
                    start = 0
                f.seek(start)
                wave[:] = f.read(frames=duration, dtype="float32")
            else:
                # ==========================================
                # 【変更点】Wrap Padding (波形の繰り返し) - Valid用
                # ==========================================
                short_wave = f.read(dtype="float32")
                repeats = (duration // n_frames) + 1
                wave = np.tile(short_wave, repeats)[:duration]

                # Validではランダム性を排除し、常に同じパディング結果になるようにする
                if is_soundscape:
                    shift_amount = (duration - n_frames) // 2
                    wave = np.roll(wave, shift_amount)

        return wave

In [24]:
class BirdTrainDataset(Dataset):
    def __init__(
        self,
        paths,
        labels,
        is_soundscape,
        sampling_rate=32_000,
        segment_sec=10,  # 実行時に CFG.context_sec (20) が渡されます
        mixup_prob=0.5,  # 【追加】Mixupを適用する確率
        mixup_alpha=0.4, # 【追加】Beta分布のパラメータ
    ):
        self.paths = paths
        self.labels = labels
        self.is_soundscape = is_soundscape
        self.sampling_rate = sampling_rate
        self.segment_sec = segment_sec
        self.mixup_prob = mixup_prob
        self.mixup_alpha = mixup_alpha
        self.resample = AT.Resample(self.sampling_rate, int(self.sampling_rate*(1+torch.randint(-5, 5, (1,))*0.01)))

    def __len__(self):
        return len(self.paths)

    def _get_single_sample(self, idx):
        """【追加】1サンプル分の波形とラベルを取得するヘルパーメソッド"""
        path = self.paths[idx]
        wave = self._load_audio_file(path, self.is_soundscape[idx])
        label = self.labels[idx].astype(np.float32)
        return wave, label

    def __getitem__(self, idx):
        # 1. メインサンプルの取得
        wave1, label1 = self._get_single_sample(idx)

        # 2. Waveform Mixup の適用
        if np.random.random() < self.mixup_prob:
            # データセット全体からランダムに別のサンプルを選択
            # (これにより Focal-Focal, Focal-Soundscape が自然な比率でMixされます)
            mix_idx = np.random.randint(0, len(self.paths))
            wave2, label2 = self._get_single_sample(mix_idx)

            # Beta分布から混合比率 lambda を生成
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)

            # 波形合成: _load_audio_file のおかげで長さが完全に一致しているため直接計算可能
            wave = (lam * wave1 + (1.0 - lam) * wave2).astype(np.float32)

            # ラベル合成: np.maximum を使い、両方の鳥が「完全に存在する(1.0)」とする (Hard Mixup)
            label = np.maximum(label1, label2)
        else:
            wave = wave1
            label = label1

        return {
            "wave": wave,
            "label": label,
        }

    def _load_audio_file(self, path, is_soundscape):
        duration = int(self.sampling_rate * self.segment_sec)

        with soundfile.SoundFile(path) as f:
            n_frames = f.frames

            if n_frames == 0:
                # 破損ファイル等で長さ0の例外処理
                return np.zeros(duration, dtype="float32")

            if n_frames >= duration:
                wave = np.zeros(duration, dtype="float32")
                if is_soundscape:
                    # soundscape split 由来は、できるだけ中央にイベントが来るように中央crop
                    start = max(0, (n_frames - duration) // 2)
                else:
                    # train_audio は従来通り random crop
                    start = np.random.randint(0, n_frames - duration + 1)

                f.seek(start)
                wave[:] = f.read(frames=duration, dtype="float32")
                # wave = self.resample(wave) # <- TBD

            else:
                # ==========================================
                # Wrap Padding (波形の繰り返し)
                # ==========================================
                short_wave = f.read(dtype="float32")
                # short_wave = self.resample(short_wave) # <- TBD
                # 必要な長さに到達するまで波形を繰り返す
                repeats = (duration // n_frames) + 1
                wave = np.tile(short_wave, repeats)[:duration]

                if not is_soundscape:
                    # train_audio の場合、繰り返しの「継ぎ目」や開始位置を毎Epochランダムにずらす
                    # これによりモデルが特定の無音パターンを暗記するのを防ぎます
                    roll_shift = np.random.randint(0, n_frames)
                    wave = np.roll(wave, roll_shift)
                else:
                    # soundscapeの場合は中央寄せのニュアンスを残すため、
                    # 元波形の中心が20秒枠の中心付近にくるようにシフト調整
                    shift_amount = (duration - n_frames) // 2
                    wave = np.roll(wave, shift_amount)

        # wave = self.apply_random_gain_db(wave, min_db=CFG.gain_min_db, max_db=CFG.gain_max_db)  # <- TBD

        return wave

    def apply_random_gain_db(self, tensor, min_db=-6, max_db=6):
        gain_db = (max_db - min_db) * torch.rand(1).item() + min_db
        gain_linear = 10 ** (gain_db / 20)
        return tensor * gain_linear

In [25]:
class BirdDataset(Dataset):
    """
    Dataset for BirdCLEF 2026 training.

    Args:
        df: DataFrame with columns [audio_id, filename, primary_label, source,
            start_sec, end_sec, label_list]
        labels: (N, n_classes) multi-label array
        is_train: training mode (enables random cropping)
        sr: audio sample rate (default 32000)
        duration: target segment duration in seconds (default 5)
        data_dir: path to data root (containing train_audio/ and train_soundscapes/)
        use_10s_soundscape: 10s mode: expand around center 5s segment with ±2.5s context
        use_noisy_aug: enable waveform gain/noise augmentation (HGNet)
        noisy_aug_prob: probability per augmentation type
        noisy_gain_db_range: gain jitter range in dB
        noisy_snr_db_range: noise SNR range in dB
        unlabeled_use_full_random: for unlabeled SC, use full file random crop
    """

    def __init__(
        self,
        df: pd.DataFrame,
        labels: np.ndarray,
        is_train: bool = True,
        sr: int = 32_000,
        duration: int = 5,
        data_dir: Path = Path("birdc"),
        use_10s_soundscape: bool = False,
        use_noisy_aug: bool = False,
        noisy_aug_prob: float = 0.5,
        noisy_gain_db_range: tuple = (-6.0, 6.0),
        noisy_snr_db_range: tuple = (3.0, 18.0),
        unlabeled_use_full_random: bool = True,
    ):
        self.df = df.reset_index(drop=True)
        self.labels = labels
        self.is_train = is_train
        self.sr = sr
        self.duration = duration
        self.audio_len = sr * duration
        self.data_dir = data_dir

        self.use_10s_soundscape = use_10s_soundscape
        self.use_noisy_aug = use_noisy_aug
        self.noisy_aug_prob = noisy_aug_prob
        self.noisy_gain_db_range = noisy_gain_db_range
        self.noisy_snr_db_range = noisy_snr_db_range
        self.unlabeled_use_full_random = unlabeled_use_full_random

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = self.labels[idx]

        if row["source"] == "focal":
            wav = self._load_focal(row)
        else:
            # Soundscape: detect unlabeled (label all zeros) for full_random mode
            is_unlabeled_sc = (
                self.unlabeled_use_full_random
                and self.is_train
                and label.sum() == 0
            )
            wav = self._load_soundscape(row, full_random=is_unlabeled_sc)

        # Optional waveform noisy augmentation (HGNet)
        if self.is_train and self.use_noisy_aug:
            wav_np = wav.numpy()
            wav_np = self._apply_noisy_aug(wav_np)
            wav = torch.from_numpy(wav_np)

        return wav, label.astype(np.float32)

    def _apply_noisy_aug(self, w):
        """Gain jitter + noise injection (HGNet style)."""
        if np.random.random() < self.noisy_aug_prob:
            w = w * (10 ** (np.random.uniform(*self.noisy_gain_db_range) / 20))
        if np.random.random() < self.noisy_aug_prob:
            sp = (w ** 2).mean()
            if sp > 1e-10:
                w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                    sp / (10 ** (np.random.uniform(*self.noisy_snr_db_range) / 10)))
        return w

    def _load_focal(self, row) -> torch.Tensor:
        path = str(self.data_dir / "train_audio" / row["filename"])
        wav, _ = torchaudio.load(path)
        return self._crop(wav[0])

    def _load_soundscape(self, row, full_random=False) -> torch.Tensor:
        path = str(self.data_dir / "train_soundscapes" / row["filename"])
        wav, _ = torchaudio.load(path)
        wav = wav[0]

        if full_random:
            # Full file: _crop will randomly select a segment
            return self._crop(wav)

        if self.use_10s_soundscape:
            # 10s mode: original 5s segment centered, expanded with ±2.5s context
            center_start = int(row["start_sec"] * self.sr)
            center_end = int(row["end_sec"] * self.sr)
            ctx = int(2.5 * self.sr)
            start = max(0, center_start - ctx)
            end = min(wav.shape[0], center_end + ctx)
            wav = wav[start:end]
        else:
            # Standard mode: precise segment
            start_sample = int(row["start_sec"] * self.sr)
            end_sample = int(row["end_sec"] * self.sr)
            wav = wav[start_sample:end_sample]

        return self._crop(wav)

    def _crop(self, wav):
        if wav.shape[0] < self.audio_len:
            return F.pad(wav, (0, self.audio_len - wav.shape[0]))
        if self.is_train:
            start = random.randint(0, wav.shape[0] - self.audio_len)
            return wav[start:start + self.audio_len]
        return wav[:self.audio_len]

#### preprocessing

In [26]:
def filt_aug(features, db_range=(-6, 6), n_band=(3, 6), min_bw=6):
    """
    Apply per-band random gain in dB on the mel frequency axis.

    Args:
        features: (B, n_freq, n_time) mel spectrogram
        db_range: (min_db, max_db) gain range per band
        n_band: (min_bands, max_bands) number of frequency bands
        min_bw: minimum bandwidth per band (in mel bins)

    Returns:
        augmented features, same shape as input
    """
    B, n_freq, _ = features.shape
    n_bands = torch.randint(n_band[0], n_band[1], (1,)).item()
    if n_bands <= 1:
        return features

    while n_freq - n_bands * min_bw + 1 < 0:
        min_bw -= 1

    bndry = torch.sort(
        torch.randint(0, n_freq - n_bands * min_bw + 1, (n_bands - 1,))
    )[0] + torch.arange(1, n_bands) * min_bw
    bndry = torch.cat([torch.tensor([0]), bndry, torch.tensor([n_freq])])

    factors = torch.rand((B, n_bands + 1), device=features.device) * (db_range[1] - db_range[0]) + db_range[0]
    freq_filt = torch.ones((B, n_freq, 1), device=features.device)

    for i in range(n_bands):
        l, r = int(bndry[i].item()), int(bndry[i + 1].item())
        for j in range(B):
            freq_filt[j, l:r, :] = torch.linspace(
                factors[j, i].item(), factors[j, i + 1].item(), r - l,
                device=features.device,
            ).unsqueeze(-1)

    return features * (10 ** (freq_filt / 10))

In [27]:
from torchvision.transforms import v2 as tvt_v2

class LogMelTransform(nn.Module):
    """
    Mel spectrogram transform WITHOUT resize (effv2s style).
    FilterAugment is applied inline during training.

    Args:
        mel_params: dict for torchaudio.transforms.MelSpectrogram
        top_db: amplitude-to-dB top_db parameter
        filt_aug_db_range: FilterAugment dB range (None = no filt_aug)
        filt_aug_n_band: FilterAugment band range
        filt_aug_min_bw: FilterAugment min bandwidth
    """

    def __init__(self, mel_params, top_db=80.0,
                 filt_aug_db_range=(-6, 6), filt_aug_n_band=(3, 6), filt_aug_min_bw=6):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(**mel_params)
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=top_db)
        self.top_db = top_db
        self.filt_aug_db_range = filt_aug_db_range
        self.filt_aug_n_band = filt_aug_n_band
        self.filt_aug_min_bw = filt_aug_min_bw

    def forward(self, wave):
        with torch.no_grad():
            lms = self.db(self.mel(wave))

        if self.training and self.filt_aug_db_range is not None:
            lms = filt_aug(lms, self.filt_aug_db_range,
                           self.filt_aug_n_band, self.filt_aug_min_bw)

        lms = torch.clamp((lms + self.top_db) / self.top_db, 0.0, 1.0)
        return lms[:, None, :, :]


class LogMelSpectrogramTransform(nn.Module):
    """
    Mel spectrogram transform WITH resize to fixed shape (effb3 / hgnet style).
    Optional FilterAugment and SpecAugment (TimeMasking + FrequencyMasking).

    Args:
        mel_params: dict for torchaudio.transforms.MelSpectrogram
        top_db: amplitude-to-dB top_db parameter
        lms_shape: (H, W) target size for Resize
        use_filt_aug: whether to apply FilterAugment
        filt_aug_db_range, filt_aug_n_band, filt_aug_min_bw: FilterAugment params
        use_spec_aug: whether to apply TimeMasking + FrequencyMasking
        time_mask_param, freq_mask_param, n_specaug: SpecAugment params
    """

    def __init__(
        self,
        mel_params,
        top_db=80.0,
        lms_shape=(128, 313),
        use_filt_aug=True,
        filt_aug_db_range=(-6, 6),
        filt_aug_n_band=(3, 6),
        filt_aug_min_bw=6,
        use_spec_aug=True,
        time_mask_param=40,
        freq_mask_param=20,
        n_specaug=2,
    ):
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(**mel_params)
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=top_db)
        self.resize = tvt_v2.Resize(size=lms_shape)
        self.top_db = top_db

        self.use_filt_aug = use_filt_aug
        self.filt_aug_db_range = filt_aug_db_range
        self.filt_aug_n_band = filt_aug_n_band
        self.filt_aug_min_bw = filt_aug_min_bw

        self.use_spec_aug = use_spec_aug
        self.n_specaug = n_specaug
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=time_mask_param)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=freq_mask_param)

    def forward(self, wave):
        with torch.no_grad():
            mel = self.mel_transform(wave)
            lms = self.db(mel)

        if self.training and self.use_filt_aug:
            lms = filt_aug(
                lms,
                db_range=self.filt_aug_db_range,
                n_band=self.filt_aug_n_band,
                min_bw=self.filt_aug_min_bw,
            )

        lms = self.resize(lms)
        lms = torch.clamp((lms + self.top_db) / self.top_db, 0.0, 1.0)
        lms = lms[:, None, :, :]

        if self.training and self.use_spec_aug:
            for _ in range(self.n_specaug):
                lms = self.freq_mask(self.time_mask(lms))

        return lms

In [28]:
# ── MixUp ──────────────────────────────────────

class MixUp(nn.Module):
    """Waveform-level mixup with union labels above theta threshold."""

    def __init__(self, alpha: float = 1.0, theta: float = 1.0):
        super().__init__()
        self.beta_dist = torch.distributions.Beta(alpha, alpha)
        self.theta = theta

    def forward(self, lms, label):
        """
        Args:
            lms: (B, C, H, W) mel spectrograms
            label: (B, n_classes) multi-label targets

        Returns:
            lms_mixed, label_mixed, lam, shuffle_idx
        """
        b = lms.shape[0]
        lam = self.beta_dist.sample((b,)).to(lms.device)
        lam = torch.maximum(lam, 1 - lam).float()
        idx = torch.randperm(b, device=lms.device)

        lms = lam[:, None, None, None] * lms + (1 - lam[:, None, None, None]) * lms[idx]
        label = lam[:, None] * label + (1 - lam[:, None]) * label[idx]
        label = label.clamp(max=1.0)
        label[label >= self.theta] = 1.0
        return lms, label, lam, idx


# ── Albumentations-based SpecAugment ────────────

def build_spec_aug(cfg):
    """
    Build albumentations pipeline for CoarseDropout + HorizontalFlip.

    cfg must have:
        mel_spectrogram_params["n_mels"], sr, duration,
        aug_flip_p, aug_dropout_frac, aug_dropout_holes, aug_dropout_p

    For 10s version, cfg may also have aug_dropout_holes as a range instead of fixed.
    """
    if not _HAS_ALBUMENTATIONS:
        raise ImportError("albumentations is required for build_spec_aug. Install with: pip install albumentations")

    n_mels = cfg.mel_spectrogram_params["n_mels"]
    hop = cfg.mel_spectrogram_params.get("hop_length", 512)
    n_time = int(cfg.sr * cfg.duration / hop) + 1
    max_h = int(n_mels * cfg.aug_dropout_frac[0])
    max_w = int(n_time * cfg.aug_dropout_frac[1])

    # Support both fixed max_holes and range (10s variant)
    num_holes = cfg.aug_dropout_holes
    if isinstance(num_holes, int):
        return A.Compose([
            A.HorizontalFlip(p=cfg.aug_flip_p),
            A.CoarseDropout(
                max_height=max_h,
                max_width=max_w,
                max_holes=num_holes,
                fill_value=0.0,
                p=cfg.aug_dropout_p,
            ),
        ])
    else:
        # num_holes is a tuple (min, max) for range-based (10s variant uses num_holes_range)
        return A.Compose([
            A.HorizontalFlip(p=cfg.aug_flip_p),
            A.CoarseDropout(
                num_holes_range=(1, num_holes),
                hole_height_range=(1, max_h),
                hole_width_range=(1, max_w),
                fill=0.0,
                p=cfg.aug_dropout_p,
            ),
        ])


def apply_spec_aug(lms, aug_fn):
    """
    Apply albumentations augmentation to a batch of spectrograms.

    Args:
        lms: (B, 1, H, W) tensor (on GPU or CPU)
        aug_fn: albumentations Compose function

    Returns:
        (B, 1, H, W) tensor, same device/dtype as input
    """
    device = lms.device
    dtype = lms.dtype
    B = lms.shape[0]

    imgs = lms.squeeze(1).cpu().numpy()
    out = np.empty_like(imgs)
    for i in range(B):
        augmented = aug_fn(image=imgs[i])["image"]
        out[i] = augmented

    return torch.from_numpy(out).unsqueeze(1).to(device=device, dtype=dtype)


# ── Torchaudio SpecAugment (HGNet variant) ─────

class SpecAugment(nn.Module):
    """
    TimeMasking-only SpecAugment (FrequencyMasking replaced by FrequencySE).

    Used by HGNet branch.
    """

    def __init__(self, freq_mask_param=6, time_mask_param=10,
                 n_freq_masks=2, n_time_masks=3):
        super().__init__()
        self.freq_mask = T.FrequencyMasking(freq_mask_param)
        self.time_mask = T.TimeMasking(time_mask_param)
        self.n_freq_masks = n_freq_masks
        self.n_time_masks = n_time_masks

    def forward(self, x):
        # Frequency masking is disabled (replaced by FrequencySE)
        # for _ in range(self.n_freq_masks):
        #     x = self.freq_mask(x)

        for _ in range(self.n_time_masks):
            x = self.time_mask(x)
        return x


# ── Waveform Noisy Augmentation (HGNet variant) ─

def apply_noisy_aug(w, aug_prob=0.5, gain_db_range=(-6.0, 6.0), noise_snr_db_range=(3.0, 18.0)):
    """
    Simple waveform augmentation: gain jitter + noise injection.

    Args:
        w: numpy array (1D waveform)
        aug_prob: probability of applying each augmentation
        gain_db_range: (min, max) gain in dB
        noise_snr_db_range: (min, max) SNR in dB for noise injection

    Returns:
        augmented waveform (numpy array, same shape)
    """
    if np.random.random() < aug_prob:
        w = w * (10 ** (np.random.uniform(*gain_db_range) / 20))
    if np.random.random() < aug_prob:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*noise_snr_db_range) / 10)))
    return w

#### model

In [29]:
class PerchTeacher:
    """
    Frozen Perch v2 via ONNX runtime.

    Args:
        onnx_path: path to the Perch ONNX model file
        prefer_cuda: use CUDAExecutionProvider if available
        sr: sample rate (default 32000)
        duration: target duration in seconds (default 5)
        perch_input_sec: Perch model's expected input length in seconds
            (default 5; set to 5 for 10s models that need center cropping)
        perch_embed_dim: expected embedding dimension (default 1536)
        sub_batch_size: sub-batch size for ONNX inference (None = no sub-batching)
    """

    def __init__(
        self,
        onnx_path,
        prefer_cuda=True,
        sr=32_000,
        duration=5,
        perch_input_sec=5,
        perch_embed_dim=1536,
        sub_batch_size=None,
    ):
        available = ort.get_available_providers()
        providers = (
            ["CUDAExecutionProvider", "CPUExecutionProvider"]
            if prefer_cuda and "CUDAExecutionProvider" in available
            else ["CPUExecutionProvider"]
        )

        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name

        self.embed_idx = next(
            (i for i, o in enumerate(self.session.get_outputs())
             if o.shape and o.shape[-1] == perch_embed_dim),
            1,
        )
        self.sr = sr
        self.duration = duration
        self.perch_input_sec = perch_input_sec
        self.perch_len = sr * perch_input_sec
        self.target_len = sr * duration
        self.sub_batch_size = sub_batch_size

        print(f"[Perch] providers={self.session.get_providers()} embed_idx={self.embed_idx}")
        if self.perch_input_sec != self.duration:
            print(f"[Perch] model input={self.perch_input_sec}s, target duration={self.duration}s → center crop")
        if sub_batch_size is not None:
            print(f"[Perch] sub_batch_size={sub_batch_size}")

    @torch.no_grad()
    def embed(self, wave):
        """
        Extract Perch embeddings from waveform.

        Args:
            wave: (B,) or (B, 1) or (B, T) waveform tensor

        Returns:
            (B, D) embedding tensor
        """
        if wave.ndim == 3:
            wave = wave.squeeze(1)

        wave = wave.float()

        # If input is longer than perch expects, center crop
        if wave.shape[1] > self.perch_len:
            start = (wave.shape[1] - self.perch_len) // 2
            wave = wave[:, start:start + self.perch_len]
        elif wave.shape[1] < self.perch_len:
            wave = F.pad(wave, (0, self.perch_len - wave.shape[1]))

        wav_np = np.ascontiguousarray(wave.cpu().numpy().astype(np.float32))

        # Sub-batched inference (for memory efficiency on large batch sizes)
        if self.sub_batch_size is not None and self.sub_batch_size < wav_np.shape[0]:
            all_outs = []
            for i in range(0, wav_np.shape[0], self.sub_batch_size):
                wav_sub = wav_np[i: i + self.sub_batch_size]
                outs = self.session.run(None, {self.input_name: wav_sub})
                all_outs.append(torch.from_numpy(outs[self.embed_idx]))
            return torch.cat(all_outs, dim=0).float()

        outs = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(outs[self.embed_idx]).float()

In [30]:
def init_layer(layer):
    """Xavier uniform init for Linear/Conv layers."""
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, "bias") and layer.bias is not None:
        layer.bias.data.fill_(0.0)


def init_bn(bn):
    """Init BatchNorm: bias=0, weight=1."""
    bn.bias.data.fill_(0.0)
    bn.weight.data.fill_(1.0)

In [31]:
class GeM1d(nn.Module):
    """
    Generalized Mean Pooling along time dimension with local kernel.

    p=1 → avg pooling, p→∞ → max pooling.
    p is learned during training.
    """

    def __init__(self, p: float = 3.0, kernel_size: int = 3, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.kernel_size = kernel_size
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        x = x.clamp(min=self.eps)
        return F.avg_pool1d(
            x.pow(self.p),
            kernel_size=self.kernel_size,
            stride=1,
            padding=self.kernel_size // 2,
        ).pow(1.0 / self.p)


# ── Attention Block (basic, effv2s / 10s) ──────

class AttBlock(nn.Module):
    """
    Basic attention-based SED head.
    clipwise = sum(softmax(att_logits) * frame_logits)

    Used by: effv2s, effv2s_10s branches.
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.att = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        init_layer(self.att)
        init_layer(self.cla)

    def forward(self, x):
        # x: (B, C, T)
        norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
        framewise = self.cla(x)
        clipwise = torch.sum(norm_att * framewise, dim=2)
        return clipwise, framewise


# ── AttBlock with BatchNorm (effb3) ────────────

class AttBlockBN(nn.Module):
    """
    Attention block with BatchNorm, used by effb3 branch.
    Also supports aux_pool for training-time auxiliary output.
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.att = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.bn_att = nn.BatchNorm1d(out_features)
        init_layer(self.att)
        init_layer(self.cla)
        init_bn(self.bn_att)

    def forward(self, x):
        norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
        framewise = self.cla(x)
        clipwise = torch.sum(norm_att * framewise, dim=2)
        return clipwise, framewise


# ── AttBlock with aux_pool (HGNet) ─────────────

class AttBlockAux(nn.Module):
    """
    Enhanced attention block with auxiliary pooling for training.
    - Inference: returns (clipwise_att, None)
    - Training: returns (clipwise_att, clipwise_aux) where aux is topk or max

    Used by: hgnet branch.
    """

    def __init__(self, in_features: int, out_features: int,
                 aux_pool: str = 'topk', k: int = 3, r: float = 1.0):
        super().__init__()
        self.att = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.bn_att = nn.BatchNorm1d(out_features)
        init_layer(self.att)
        init_layer(self.cla)
        init_bn(self.bn_att)

        self.aux_pool = aux_pool
        self.k = k
        self.r = r

    def forward(self, x):
        norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
        framewise = self.cla(x)
        clipwise_att = torch.sum(norm_att * framewise, dim=2)

        if not self.training:
            return clipwise_att, None

        if self.aux_pool == 'topk':
            actual_k = min(self.k, framewise.size(-1))
            clipwise_aux = framewise.topk(actual_k, dim=2)[0].mean(dim=2)
        else:
            clipwise_aux = framewise.max(dim=2)[0]

        return clipwise_att, clipwise_aux


# ── Distill Head ───────────────────────────────

class DistillHead(nn.Module):
    """Project backbone feature map to Perch embedding space."""

    def __init__(self, backbone_dim: int, embed_dim: int = 1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
        init_layer(self.proj)

    def forward(self, feat_map):
        # feat_map: (B, C, H, W)
        gap = feat_map.mean(dim=[2, 3])  # (B, C)
        return self.proj(gap)


# ── Channel Attention (HGNet) ──────────────────

class ChannelAttention(nn.Module):
    """SE-like channel attention along feature dimension."""

    def __init__(self, dim: int = 2048, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(dim // reduction, dim, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, T)
        avg_out = self.fc(self.avg_pool(x).squeeze(-1))
        max_out = self.fc(self.max_pool(x).squeeze(-1))
        out = self.sigmoid(avg_out + max_out).unsqueeze(-1)
        return x * out


# ── Frequency SE (HGNet) ───────────────────────

class FrequencySE(nn.Module):
    """
    Squeeze-and-Excitation along frequency axis.
    Replaces FrequencyMasking in SpecAugment.
    """

    def __init__(self, n_mels: int = 128, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d((n_mels, 1))
        self.fc = nn.Sequential(
            nn.Linear(n_mels, n_mels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(n_mels // reduction, n_mels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, C, H, W)
        b, c, h, w = x.size()
        y = self.avg_pool(x).squeeze(-1).squeeze(1)
        y = self.fc(y).view(b, 1, h, 1)
        return x * y.expand_as(x)

In [32]:
class BirdModel(nn.Module):
    """
    BirdCLEF SED model with timm backbone + attention head + optional Perch distill head.

    Args:
        model_name: timm model name
        pretrained: use timm online pretrained weights
        pretrain_ckpt: path to custom pretrained checkpoint (None = skip)
        drop_path_rate, drop_rate: backbone stochastic depth / dropout
        num_classes: number of output classes (default 234)
        head_dropout: dropout rate in classification head
        n_mels: number of mel frequency bins
        use_hgnet: enable HGNet-specific modules (ChannelAttention, FrequencySE, AttBlockAux)
        use_effb3: enable effb3-specific modules (AttBlockBN, detach_cls_branch option)
        use_distill: create DistillHead for Perch embedding distillation
        embed_dim: Perch embedding dimension
        detach_cls_branch: stop gradient from distill head to cls branch (effb3 option)
    """

    def __init__(
        self,
        model_name,
        pretrained=True,
        pretrain_ckpt=None,
        drop_path_rate=0.0,
        drop_rate=0.2,
        num_classes=234,
        head_dropout=0.5,
        n_mels=128,
        use_hgnet=False,
        use_effb3=False,
        use_distill=True,
        embed_dim=1536,
        detach_cls_branch=False,
    ):
        super().__init__()

        self.use_hgnet = use_hgnet
        self.use_effb3 = use_effb3
        self.use_distill = use_distill
        self.detach_cls_branch = detach_cls_branch

        # --- BN0 ---
        self.bn0 = nn.BatchNorm2d(n_mels)
        init_bn(self.bn0)

        # --- Backbone ---
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            in_chans=1,
            global_pool="",
            num_classes=0,
            drop_path_rate=drop_path_rate,
            drop_rate=drop_rate,
        )

        # Load custom pretrain checkpoint (XLS pretrained weights for effv2s)
        if pretrain_ckpt is not None and pretrain_ckpt:
            ckpt_path = Path(pretrain_ckpt)
            if not ckpt_path.exists():
                print(f"[Pretrain] WARNING: checkpoint not found at {ckpt_path}, "
                      f"using timm pretrained={pretrained} instead")
            else:
                ckpt = torch.load(ckpt_path, map_location="cpu")
                missing, unexpected = self.backbone.load_state_dict(ckpt, strict=False)
                print(f"[Pretrain] loaded from {ckpt_path}")
                print(f"[Pretrain] missing={len(missing)}, unexpected={len(unexpected)}")

        # --- Detect backbone output dim ---
        with torch.no_grad():
            self.backbone_dim = self.backbone(torch.randn(1, 1, n_mels, n_mels)).shape[1]

        # --- GeM pooling ---
        self.gem = GeM1d(p=3.0, kernel_size=3)

        # --- FC + AttBlock ---
        self.fc1 = nn.Linear(self.backbone_dim, self.backbone_dim, bias=True)
        init_layer(self.fc1)

        if use_hgnet:
            self.att_block = AttBlockAux(self.backbone_dim, num_classes)
        elif use_effb3:
            self.att_block = AttBlockBN(self.backbone_dim, num_classes)
        else:
            self.att_block = AttBlock(self.backbone_dim, num_classes)

        self.dropout = nn.Dropout(head_dropout)

        # --- Distill head ---
        if CFG.use_distill:
            self.distill_head = DistillHead(self.backbone_dim, embed_dim)

        # --- HGNet-specific modules (with random fork for deterministic init) ---
        if use_hgnet:
            with torch.random.fork_rng():
                torch.manual_seed(999)
                self.channel_att = ChannelAttention()
                self.freq_se = FrequencySE(n_mels=n_mels, reduction=16)

    def forward(self, x, return_distill=False):
        # --- BN0 + freq SE ---
        x = x.transpose(1, 2)
        x = self.bn0(x)
        x = x.transpose(1, 2)

        if self.use_hgnet:
            x = self.freq_se(x)

        feat_map = self.backbone(x)  # (B, C, H, W)

        # --- Distill embedding ---
        distill_emb = None
        if return_distill and self.use_distill:
            distill_emb = self.distill_head(feat_map)

        # --- Detach for cls branch (effb3 option) ---
        if self.use_distill and self.training and self.detach_cls_branch:
            feat = feat_map.detach()
        else:
            feat = feat_map

        # --- Feature processing ---
        feat = feat.mean(dim=2)  # (B, C, W)

        if self.use_hgnet:
            feat = self.channel_att(feat)

        feat = self.gem(feat)
        feat = self.dropout(feat)
        feat = feat.transpose(1, 2)
        feat = F.relu_(self.fc1(feat))
        feat = feat.transpose(1, 2)
        feat = self.dropout(feat)

        # --- Attention head ---
        clipwise, framewise = self.att_block(feat)

        if return_distill:
            return clipwise, framewise, distill_emb
        return clipwise, framewise

In [33]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class HybridLSEAttentionHead(nn.Module):
    """
    Parallel Hybrid Head: LSE Pooling (Main) + Attention Pooling (Auxiliary)
    """
    def __init__(
        self,
        num_features: int,
        num_classes: int,
        dropout: float = 0.2,
        smooth_power: float = 1.0,
    ):
        super().__init__()
        self.smooth_power = float(smooth_power)

        # 1. 時間軸ごとのロジット (鳥の存在確率)
        self.fc_logits = nn.Sequential(
            nn.Linear(num_features, num_features),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(num_features, num_classes),
        )

        # 2. 時間軸ごとのアテンション (文脈の重要度)
        self.fc_attention = nn.Sequential(
            nn.Linear(num_features, num_features),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(num_features, num_classes),
        )

    def forward(self, h):
        """
        h: (B, C, F, T) -> CNNバックボーンの出力
        """
        # 周波数軸を潰して時間軸とチャンネル軸を入れ替え: (B, T, C)
        h = torch.mean(h, dim=2)      
        h = h.transpose(1, 2)         

        # 1. Frame-wise Logits (時間ごとの予測)
        frame_logits = self.fc_logits(h)  # (B, T, num_classes)
        T = frame_logits.size(1)

        # 2. LSE Pooling (主ロス用)
        # torch.logsumexp を使うことで AMP(fp16) 環境でも NaN や Inf を防ぐ安定した計算
        r = self.smooth_power
        lse_logits = (torch.logsumexp(frame_logits * r, dim=1) - math.log(T)) / r

        # 3. Attention Pooling (補助ロス用)
        # tanh で -1.0〜1.0 にクリップしてから Softmax をかけることで極端な重みの集中を防ぐ
        frame_attention = torch.tanh(self.fc_attention(h))
        attn_weights = F.softmax(frame_attention, dim=1)
        
        # 確率ではなく、生ロジットを加重平均する
        attn_logits = torch.sum(frame_logits * attn_weights, dim=1)

        # 辞書型で返す
        return {
            "lse_logits": lse_logits,   # (B, num_classes) - Main Loss
            "attn_logits": attn_logits, # (B, num_classes) - Aux Loss
            "frame_logits": frame_logits, # (B, T, num_classes) - Inference
            "attn_weights": attn_weights # ★ 可視化のためにこの1行を追加
        }

class LSEModel(nn.Module):
    def __init__(
        self,
        model_name: str,
        pretrained: bool,
        drop_path_rate: float = 0.0,
        num_classes: int = 234,
        head_dropout: float = 0.2,
        input_shape=(256, 512),
        smooth_power: float = 1.0,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            in_chans=1,
            global_pool="",
            num_classes=0,
            drop_path_rate=drop_path_rate,
        )

        dummy_input = torch.randn(1, 1, *input_shape)
        with torch.no_grad():
            dummy_output = self.backbone(dummy_input)

        num_features = dummy_output.shape[1]

        # 新しいハイブリッドヘッドをインスタンス化
        self.head = HybridLSEAttentionHead(
            num_features=num_features,
            num_classes=num_classes,
            dropout=head_dropout,
            smooth_power=smooth_power,
        )
        
    def forward_inference(self, x):
        """
        推論専用メソッド: 60秒のフル音声を一括で処理
        """
        h = self.backbone(x)          # (B, C, F, T_full)
        h = torch.mean(h, dim=2)      # (B, C, T_full)
        h = h.transpose(1, 2)         # (B, T_full, C)

        # Attentionは通さず、直接ロジットだけを計算する (高速化・メモリ節約)
        timewise_logits = self.head.fc_logits(h)  # (B, T_full, num_classes)
        
        # 端数フレームを切り捨てて (B, 96, num_classes) で返す
        return timewise_logits[:, :96, :]
        
    def forward(self, x):
        """
        学習・バリデーション用: 辞書を返す
        """
        h = self.backbone(x)
        outputs = self.head(h)
        return outputs

In [34]:
class CustomBCEWithLogitsLoss(nn.BCEWithLogitsLoss):
    """BCEWithLogitsLoss for SEDModel."""

    def __init__(self, timewise_weight: float=0.5, temperature: float=1.0, **kwargs):
        """"""
        super().__init__(**kwargs)
        self.timewise_weight = timewise_weight
        self.temperature     = temperature

    def forward(self, logits, timewise_logits, label):
        """
        logits         : (B, N_CLASSES)
        timewise_logits: (B, Time, N_CLASSES)
        """
        loss = super().forward(logits, label)
        logits_timeaxis_lse = self.temperature * (
            torch.logsumexp(timewise_logits / self.temperature, axis=1)
            - math.log(timewise_logits.shape[1])
        )
        loss_timeaxis_lse = super().forward(logits_timeaxis_lse, label)

        return (1 - self.timewise_weight) * loss + self.timewise_weight * loss_timeaxis_lse

#### utility

In [35]:
def to_device(tensors, device: torch.device, *args, **kwargs):
    if isinstance(tensors, tuple):
        return tuple(t.to(device, *args, **kwargs) for t in tensors)
    elif isinstance(tensors, list): # ★リスト型の場合の処理を追加
        return [t.to(device, *args, **kwargs) for t in tensors]
    elif isinstance(tensors, dict):
        return {k: t.to(device, *args, **kwargs) for k, t in tensors.items()}
    else:
        return tensors.to(device, *args, **kwargs)

In [36]:
from torch.utils.data import WeightedRandomSampler

def upsample_rare(df, rare_labels, label_col="primary_label", min_count=100, random_state=42):
    """
    rare_labels に含まれるクラスについて、label_col の件数が min_count 未満なら
    復元抽出で min_count まで増やす
    """
    dfs = [df]

    for label in rare_labels:
        df_label = df[df[label_col] == label]
        n = len(df_label)

        if n == 0 or n >= min_count:
            continue

        n_add = min_count - n
        df_add = df_label.sample(
            n=n_add,
            replace=True,
            random_state=random_state,
        )
        dfs.append(df_add)

    out = pd.concat(dfs, axis=0, ignore_index=True)
    return out


def build_primary_label_sample_weights(df, label_col="primary_label", gamma=-0.5):
    """
    各サンプルに対して primary_label ベースの重みを返す
    w_i = (c_i / sum_j c_j)^gamma
    """
    class_counts = df[label_col].value_counts()
    class_freq = class_counts / class_counts.sum()
    class_weights = class_freq.pow(gamma)
    sample_weights = df[label_col].map(class_weights).astype(np.float32).values
    return sample_weights, class_counts, class_weights

In [37]:
def get_data_loader(train_df, fold_id, device=None):
    # foldで分割
    trn_df = train_df.query("fold != @fold_id").reset_index(drop=True).copy()
    val_df = train_df.query("fold == @fold_id").reset_index(drop=True).copy()
    
    # -------------------------
    # fold split
    # -------------------------
    # trn_idxs = trn_df.query("fold != @fold_id").index.values
    # val_idxs = val_df.query("fold == @fold_id").index.values

    # -------------------------
    # upsample rare classes in TRAIN only
    # -------------------------
    base_counts = trn_df["primary_label"].value_counts()
    rare = base_counts[base_counts < CFG.upsample_min_count].index

    trn_df = upsample_rare(
        trn_df,
        rare_labels=rare,
        label_col="primary_label",
        min_count=CFG.upsample_min_count,
        random_state=RANDOM_SEED,
    )
    
    # -------------------------
    # build paths / labels
    # -------------------------
    file_paths = trn_df["file_path"].values.tolist()
    labels_arr = trn_df[CLASSES].values
    is_soundscape_arr = trn_df["filename"].str.lower().str.endswith(".wav").values

    # trn_paths  = [file_paths[idx] for idx in trn_idxs]
    # trn_labels = [labels_arr[idx] for idx in trn_idxs]
    # trn_is_ss  = [is_soundscape_arr[idx] for idx in trn_idxs]
    trn_paths = trn_df["file_path"].values.tolist()
    trn_labels = trn_df[CLASSES].values
    trn_is_ss = trn_df["filename"].str.lower().str.endswith(".wav").values

    # val_paths  = [file_paths[idx] for idx in val_idxs]
    # val_labels = [labels_arr[idx] for idx in val_idxs]
    # val_is_ss  = [is_soundscape_arr[idx] for idx in val_idxs]
    val_paths = val_df["file_path"].values.tolist()
    val_labels = val_df[CLASSES].values
    val_is_ss = val_df["filename"].str.lower().str.endswith(".wav").values

    # -------------------------
    # datasets
    # -------------------------
    trn_dataset = BirdTrainDataset(
        trn_paths, trn_labels, trn_is_ss,
        segment_sec=CFG.segment_sec,
    )
    val_dataset = BirdValidDataset(
        val_paths, val_labels, val_is_ss,
        segment_sec=CFG.segment_sec,
    )

    # -------------------------
    # weighted sampler
    # -------------------------
    if CFG.use_weighted_sampler:
        sample_weights, upsampled_counts, class_weights = build_primary_label_sample_weights(
            trn_df,
            label_col="primary_label",
            gamma=CFG.sampling_gamma,
        )

        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )

        trn_loader = DataLoader(
            trn_dataset,
            batch_size=CFG.batch_size, sampler=sampler, shuffle=False,   # sampler 使用時は False
            drop_last=True, num_workers=16, persistent_workers=True, pin_memory=True,
            prefetch_factor=2
        )
    else:
        trn_loader = DataLoader(
            trn_dataset,
            batch_size=CFG.batch_size, shuffle=True, drop_last=True,
            num_workers=16, persistent_workers=True, pin_memory=True, prefetch_factor=2,
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG.batch_size, shuffle=False, drop_last=False,
        num_workers=16, persistent_workers=True, pin_memory=True, prefetch_factor=2,
    )

    return trn_loader, val_loader

In [38]:
def parse_soundscape_labels(data_dir: Path) -> pd.DataFrame:
    """
    Parse train_soundscapes_labels.csv into structured format.

    Returns DataFrame with columns:
        filename, start, end, label_list, start_sec, end_sec, primary_label, source
    """
    raw = pd.read_csv(data_dir / "train_soundscapes_labels.csv")

    sc = (
        raw.groupby(["filename", "start", "end"])["primary_label"]
        .apply(
            lambda s: sorted({
                lbl.strip()
                for x in s if pd.notna(x)
                for lbl in str(x).split(";")
                if lbl.strip()
            })
        )
        .reset_index(name="label_list")
    )

    sc["start_sec"] = pd.to_timedelta(sc["start"]).dt.total_seconds().astype(int)
    sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
    sc["primary_label"] = sc["label_list"].apply(lambda x: x[0] if x else None)
    sc["source"] = "soundscape"

    return sc


def build_combined_df(
    data_dir: Path,
    taxonomy: pd.DataFrame,
    rating_threshold: float = 2.5,
    use_unlabeled_sc: bool = False,
    unlabeled_chunks_per_file: int = 2,
    min_samples_per_class: int = 0,
) -> tuple:
    """
    Build the combined training DataFrame with focal + soundscape (+ unlabeled).

    Args:
        data_dir: root data directory
        taxonomy: taxonomy DataFrame (primary_label column)
        rating_threshold: minimum rating for focal clips (XC collection)
        use_unlabeled_sc: include unlabeled soundscapes
        unlabeled_chunks_per_file: K chunks per unlabeled file
        min_samples_per_class: if >0, oversample rare species (HGNet)

    Returns:
        (df, labels_list, labels_arr)
    """
    labels_list = taxonomy["primary_label"].values.tolist()
    label2idx = {l: i for i, l in enumerate(labels_list)}

    # --- Focal data ---
    train_csv = pd.read_csv(data_dir / "train.csv")
    before = len(train_csv)
    train_csv = train_csv[
        (train_csv["rating"] >= rating_threshold)
        | (train_csv["collection"] == "iNat")
    ].reset_index(drop=True)
    print(f"Rating filter ({rating_threshold}): "
          f"{before} → {len(train_csv)} (removed {before - len(train_csv)})")

    focal = train_csv[["filename", "primary_label", "secondary_labels"]].copy()
    focal["source"] = "focal"
    focal["start_sec"] = 0
    focal["end_sec"] = 0
    focal["label_list"] = [
        [pl] + ast.literal_eval(sls)
        for pl, sls in focal[["primary_label", "secondary_labels"]].values
    ]
    focal["audio_id"] = focal["filename"].str.replace(".ogg", "", regex=False)

    # --- Labeled soundscapes ---
    ss_path = data_dir / "train_soundscapes_labels.csv"
    if ss_path.exists():
        sc = parse_soundscape_labels(data_dir)
        sc["audio_id"] = sc["filename"].str.replace(".wav", "", regex=False)
        sc["audio_id"] = sc["audio_id"].str.replace(".ogg", "", regex=False)
    else:
        sc = pd.DataFrame(columns=focal.columns)

    # --- Unlabeled soundscapes ---
    if use_unlabeled_sc:
        labeled_filenames = set(sc["filename"].values) if not sc.empty else set()
        unl_df = _build_unlabeled_sc_df(data_dir, labeled_filenames, unlabeled_chunks_per_file)
    else:
        unl_df = pd.DataFrame()

    # --- Merge ---
    keep_cols = ["audio_id", "filename", "primary_label", "source",
                 "start_sec", "end_sec", "label_list"]

    dfs = [focal[keep_cols], sc[keep_cols]]
    if len(unl_df) > 0:
        dfs.append(unl_df[keep_cols])

    df = pd.concat(dfs, ignore_index=True)

    # --- Label array ---
    labels_arr = np.zeros((len(df), len(labels_list)), dtype=np.float32)
    for i, ll in enumerate(df["label_list"].values):
        for l in ll:
            if l in label2idx:
                labels_arr[i, label2idx[l]] = 1.0

    # --- Oversampling for rare species (HGNet) ---
    if min_samples_per_class > 0:
        print('[Execute Upsampling]')
        focal_mask = df["source"] == "focal"
        counts = df.loc[focal_mask, "primary_label"].value_counts()
        rare_species = counts[counts < min_samples_per_class].index.tolist()

        extra_dfs = []
        extra_labels = []

        for sp in rare_species:
            sp_mask = focal_mask & (df["primary_label"] == sp)
            sp_df = df[sp_mask]
            sp_labels = labels_arr[sp_mask.values]

            n_copies = int(np.ceil(min_samples_per_class / len(sp_df))) - 1
            for _ in range(max(0, n_copies)):
                extra_dfs.append(sp_df.copy())
                extra_labels.append(sp_labels.copy())

        if extra_dfs:
            n_before = len(df)
            df = pd.concat([df] + extra_dfs, ignore_index=True)
            labels_arr = np.concatenate([labels_arr] + extra_labels, axis=0)
            print(f"[Upsample] {len(rare_species)} rare species: "
                  f"{n_before} -> {len(df)} samples (+{len(df) - n_before} copies)")
    else:
        print("[No execute upsampling]")

    return df, labels_list, labels_arr


def _build_unlabeled_sc_df(data_dir: Path, labeled_filenames: set, K: int) -> pd.DataFrame:
    """
    Build DataFrame for unlabeled soundscapes.

    Args:
        data_dir: root data directory
        labeled_filenames: set of soundscape filenames that are already labeled
        K: number of chunks per file

    Returns:
        DataFrame with source="soundscape", label_list=[]
    """
    soundscapes_dir = data_dir / "train_soundscapes"
    if not soundscapes_dir.exists():
        print(f"[Unlabeled] {soundscapes_dir} does not exist, skipping")
        return pd.DataFrame()

    all_sc_files = sorted(soundscapes_dir.glob("*.ogg"))
    if not all_sc_files:
        all_sc_files = sorted(soundscapes_dir.glob("*.wav"))

    unlabeled_files = [f for f in all_sc_files if f.name not in labeled_filenames]
    print(f"[Unlabeled] Found {len(all_sc_files)} soundscape files, "
          f"{len(unlabeled_files)} unlabeled")

    unl_rows = []
    for fp in unlabeled_files:
        for k in range(K):
            unl_rows.append({
                "audio_id": fp.stem,
                "filename": fp.name,
                "primary_label": None,
                "source": "soundscape",
                "start_sec": k * 5,
                "end_sec": k * 5 + 5,
                "label_list": [],
            })

    unl_df = pd.DataFrame(unl_rows)
    print(f"[Unlabeled] Added {len(unl_df)} chunks (K={K})")
    return unl_df


def assign_folds(df: pd.DataFrame, labels_arr: np.ndarray,
                 n_folds: int, seed: int) -> pd.DataFrame:
    """
    Assign cross-validation folds using MultilabelStratifiedKFold.

    Only focal samples are split into folds; all soundscapes go to training.

    Folds are assigned by audio_id (group-level) to prevent leakage.
    """
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

    df = df.copy()
    df["fold"] = -1
    focal_mask = df["source"] == "focal"

    groups = df.loc[focal_mask, "audio_id"].values
    unique_groups = np.unique(groups)
    group_map = {g: i for i, g in tqdm(enumerate(unique_groups))}

    group_labels = np.zeros((len(unique_groups), labels_arr.shape[1]), dtype=np.float32)
    for i, g in tqdm(enumerate(unique_groups)):
        group_labels[i] = labels_arr[focal_mask.values][groups == g].max(axis=0)

    mskf = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    group_folds = np.full(len(unique_groups), -1, dtype=int)
    for fold_id, (_, val_idx) in enumerate(mskf.split(group_labels, group_labels)):
        group_folds[val_idx] = fold_id

    df.loc[focal_mask, "fold"] = [group_folds[group_map[g]] for g in groups]
    return df


def build_loaders(df, labels_arr, fold_id, cfg, BirdDataset):
    """
    Build train and validation DataLoaders.

    Train: all focal not in fold + all soundscapes (including unlabeled)
    Val: focal in fold only
    """
    focal_trn = (df["source"] == "focal") & (df["fold"] != fold_id)
    focal_val = (df["source"] == "focal") & (df["fold"] == fold_id)
    sc_trn = df["source"] == "soundscape"

    trn_mask = focal_trn | sc_trn
    val_mask = focal_val

    trn_ds = BirdDataset(df[trn_mask], labels_arr[trn_mask.values], is_train=True)
    val_ds = BirdDataset(df[val_mask], labels_arr[val_mask.values], is_train=False)

    from torch.utils.data import DataLoader

    trn_loader = DataLoader(
        trn_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )
    return trn_loader, val_loader

### run training

In [39]:
class CFG:
    FOLDS        = [0] #, 1, 2, 3] # [0] # [0, 1, 2, 3]
    max_epoch    = 35 # 20
    warmup_epoch = 5
    batch_size   = 64   # OOMなら 32 に下げる
    lr           = 8e-4 # 5e-04
    weight_decay = 6e-4 # 1e-04

    # for model
    model_name      = "hgnetv2_b0.ssld_stage2_ft_in1k"
    pretrained      = True
    drop_path_rate  = 0.15 # 0.0
    drop_rate       = 0.2
    head_dropout    = 0.45
    lse_temperature = 1.0

    is_hgnet          = True
    use_effb3         = True
    has_ema           = False
    use_distill       = True
    detach_cls_branch = False
    embed_dim         = 1536

    # upsampling
    sampling_gamma = -0.5   # 画像の1行目に対応
    upsample_min_count = 100
    use_weighted_sampler = False

    # context setting
    context_sec = 5 # 10
    target_sec  = 5
    segment_sec = context_sec   # 既存コード互換のため残す

    # for log mel spectrogram transform
    mel_spectrogram_params = dict(
        sample_rate= 32_000,
        n_fft      = 2048,
        win_length = 1024, # hop_lengthに合わせて少し広げる
        hop_length = 512, # 625,  # ★重要: 32000 / 625 = 51.2 fps
        f_min      = 20,
        f_max      = 16000,
        n_mels     = 128,
        power      = 2.0,
        center     = True,
        pad_mode   = "reflect",
        norm       = "slaney",
        mel_scale  = "htk",
    )
    
    # 10秒入力で5秒時と近い時間解像度を維持
    lms_shape = (128, 313) # (256, 512)
    top_db = 80.0

    # other
    mixup = dict(alpha=1.0, theta=0.8)
    use_amp = True
    use_dp = False

    # center-weighted pooling
    outer_weight = 1.0 # 0.35
    smooth_power = 1.0 # 1.0

    # --- SpecAugment ---
    FREQ_MASK_PARAM = 10
    TIME_MASK_PARAM = 10
    NUM_FREQ_MASKS  = 1
    NUM_TIME_MASKS  = 2
    gain_min_db     = -9 
    gain_max_db     = 9

    num_workers               = 16
    n_folds                   = 4
    seed                      = 520
    sr                        = 32_000
    rating_threshold          = 2.5
    duration                  = 5
    is_10s                    = duration == 10
    data_dir                  = Path("/kaggle/input/competitions/birdclef-2026")
    use_filt_aug              = True
    use_noisy_aug             = True
    unlabeled_use_full_random = False # True # <- stage2 /3 用
    use_unlabeled_sc          = False
    unlabeled_chunks_per_file = 2
    AUG_PROB                  = 0.5
    filt_aug_db_range         = (-6, 6)
    filt_aug_n_band           = (3, 6)
    filt_aug_min_bw           = 6
    AUG_GAIN_DB_RANGE         = (-6.0, 6.0)
    AUG_NOISE_SNR_DB_RANGE    = (3.0, 18.0)
    loss_weights              = (0.6, 0.4)
    add_ss_3x                 = False
    min_samples_per_class     = 66
    use_perch_distill         = True
    distill_loss_type         = "cosine"
    alpha_distill             = 0.1
    perch_hf_repo             = "justinchuby/Perch-onnx"
    perch_hf_file             = "perch_v2_no_dft.onnx" 
    perch_embed_dim           = 1536
    mixup_alpha               = 1.0
    mixup_theta               = 0.8
    grad_clip_norm            = None
    sd_alpha                  = 0.5
    sd_threshold              = 0.4
    sd_power                  = 2.0

    # --- MixUp Control ---
    # 'waveform', 'spectrogram', 'both', 'none' のいずれかを指定
    mixup_mode                = 'waveform'

    # Waveform MixUp 用のパラメータ
    wave_mixup_prob           = 0.5
    wave_mixup_alpha          = 0.4

    # Spectrogram MixUp 用のパラメータ
    spec_mixup_alpha          = 1.0
    spec_mixup_theta          = 0.8

    # ema_decay                 = 0.999

In [40]:
class EarlyStoppingAndTopKSaver:
    def __init__(self, fold_id, patience=5, top_k=3, out_dir="./"):
        self.fold_id = fold_id
        self.patience = patience
        self.top_k = top_k
        self.out_dir = out_dir
        
        self.counter = 0
        self.best_score = -np.inf
        self.early_stop = False
        
        # 保存されているTop-Kのリスト: [(score, epoch, path), ...]
        self.top_k_checkpoints = [] 

    def __call__(self, epoch, score, model_state_dict):
        # --- Top-K の保存処理 ---
        save_path = os.path.join(self.out_dir, f"fold{self.fold_id}_epoch{epoch}.pt")
        
        # リストに追加してスコアの降順でソート
        self.top_k_checkpoints.append((score, epoch, save_path))
        self.top_k_checkpoints.sort(key=lambda x: x[0], reverse=True)
        
        # Top-Kに入った場合はファイルを保存
        if (score, epoch, save_path) in self.top_k_checkpoints[:self.top_k]:
            torch.save(model_state_dict, save_path)
            
        # Top-Kから漏れた（K+1番目になった）古いファイルを削除
        if len(self.top_k_checkpoints) > self.top_k:
            removed_item = self.top_k_checkpoints.pop(-1)
            if os.path.exists(removed_item[2]):
                os.remove(removed_item[2])

        # --- Early Stopping の判定処理 ---
        if score > self.best_score:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

In [41]:
def average_top_k_checkpoints(stopper_instance, final_save_path):
    print(f"Averaging top {len(stopper_instance.top_k_checkpoints)} models...")
    
    # Top-Kのファイルパスを取得
    paths = [item[2] for item in stopper_instance.top_k_checkpoints]
    
    # すべての重みをCPUにロード
    state_dicts = [torch.load(p, map_location="cpu") for p in paths]
    
    # 平均化
    avg_state_dict = {}
    for key in state_dicts[0].keys():
        avg_state_dict[key] = sum(sd[key] for sd in state_dicts) / float(len(state_dicts))
        
    # 平均化されたモデルを保存
    torch.save(avg_state_dict, final_save_path)
    print(f"Saved averaged model to {final_save_path}")

In [42]:
class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=CFG.FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=CFG.TIME_MASK_PARAM)

    def forward(self, mel):
        for _ in range(CFG.NUM_FREQ_MASKS):
            mel = self.freq_mask(mel)
        for _ in range(CFG.NUM_TIME_MASKS):
            mel = self.time_mask(mel)
        return mel

In [43]:
def make_grad_scaler(device, enabled=True):
    """Create a GradScaler compatible with both old and new PyTorch APIs."""
    enabled = bool(enabled and device.type == "cuda")
    try:
        return torch.GradScaler(device.type, enabled=enabled)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=enabled)

In [44]:
def compute_distill_loss(pred, target, loss_type="cosine"):
    """
    Distillation loss between student and teacher embeddings.

    Args:
        pred: student embedding (B, D)
        target: teacher embedding (B, D), will be detached
        loss_type: "cosine", "mse", or "norm_mse"

    Returns:
        scalar loss
    """
    target = target.detach()

    if loss_type == "cosine":
        pred = F.normalize(pred.float(), dim=1)
        target = F.normalize(target.float(), dim=1)
        return 1.0 - (pred * target).sum(dim=1).mean()

    elif loss_type == "mse":
        return F.mse_loss(pred.float(), target.float())

    elif loss_type == "norm_mse":
        pred = F.normalize(pred.float(), dim=1)
        target = F.normalize(target.float(), dim=1)
        return F.mse_loss(pred, target)

    else:
        raise ValueError(f"Unknown distill loss type: {loss_type}")

In [45]:
def train_one_fold(train_df, fold_id, device, output_dir=None, dry_run=False):
    if output_dir == None:
        output_dir = Path.cwd()
    else:
        output_dir.mkdir(exist_ok=True)

    if dry_run:
        train_df = train_df.head(200)
        CFG.max_epoch = 10

    print(f"[training fold {fold_id}]")
    print(f"train: {len(train_df.query("fold != @fold_id"))}, valid: {len(train_df.query("fold == @fold_id"))}")

    set_random_seed(RANDOM_SEED)

    # --- Build loaders ---
    focal_trn = (train_df["source"] == "focal") & (train_df["fold"] != fold_id)
    focal_val = (train_df["source"] == "focal") & (train_df["fold"] == fold_id)
    trn_mask = focal_trn | (train_df["source"] == "soundscape")

    # Print training set composition
    n_focal = focal_trn.sum()
    n_sc_total = (train_df["source"] == "soundscape").sum()
    n_unl = ((train_df["source"] == "soundscape") & (train_df["label_list"].apply(len) == 0)).sum()
    n_labeled_sc = n_sc_total - n_unl
    print(f"\n[Fold {fold_id} training set composition]")
    print(f"  focal:        {n_focal:>7d}")
    if n_labeled_sc > 0 or n_unl > 0:
        print(f"  labeled_sc:   {n_labeled_sc:>7d}")
        print(f"  unlabeled_sc: {n_unl:>7d}")
    print(f"  total train:  {n_focal + n_sc_total:>7d}")
    print(f"  val (focal):  {focal_val.sum():>7d}")

    # --- Dataset ---
    # trn_loader, val_loader = get_data_loader(train_df, fold_id, device)
    ds_kwargs = dict(
        sr=CFG.sr, duration=CFG.duration, data_dir=CFG.data_dir,
    )
    if CFG.is_hgnet:
        ds_kwargs.update(
            use_noisy_aug=CFG.use_noisy_aug,
            noisy_aug_prob=CFG.AUG_PROB,
            noisy_gain_db_range=CFG.AUG_GAIN_DB_RANGE,
            noisy_snr_db_range=CFG.AUG_NOISE_SNR_DB_RANGE,
        )
    if CFG.is_10s:
        ds_kwargs['use_10s_soundscape'] = True
    if hasattr(CFG, 'unlabeled_use_full_random'):
        print("use unlabeled_use_full_random")
        ds_kwargs['unlabeled_use_full_random'] = CFG.unlabeled_use_full_random

    # 1. Dataset側 (Waveform MixUp) の制御
    trn_loader = DataLoader(
        BirdDataset(train_df[trn_mask], 
                    labels_arr[trn_mask.values],
                    is_train=True,
                    **ds_kwargs),
        batch_size=CFG.batch_size, shuffle=True, drop_last=True,
        num_workers=CFG.num_workers, pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )
    val_loader = DataLoader(
        BirdDataset(train_df[focal_val], 
                    labels_arr[focal_val.values], 
                    is_train=False, 
                    **ds_kwargs),
        batch_size=CFG.batch_size, shuffle=False, drop_last=False,
        num_workers=CFG.num_workers, pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )
    
    # --- LogMel transform ---
    is_10s_sc = hasattr(CFG, 'mel_spectrogram_params') and CFG.mel_spectrogram_params.get('n_mels') == 128 and CFG.duration == 10

    if hasattr(CFG, 'lms_shape'):
        # effb3 / hgnet: Resize-based transform
        print("Resize-based transform")
        lms_transform = LogMelSpectrogramTransform(
            CFG.mel_spectrogram_params, CFG.top_db, CFG.lms_shape,
            use_filt_aug=getattr(CFG, 'use_filt_aug', True),
            filt_aug_db_range=CFG.filt_aug_db_range,
            filt_aug_n_band=CFG.filt_aug_n_band,
            filt_aug_min_bw=CFG.filt_aug_min_bw,
            use_spec_aug=getattr(CFG, 'use_spec_aug', False),
            time_mask_param=getattr(CFG, 'time_mask_param', 40),
            freq_mask_param=getattr(CFG, 'freq_mask_param', 20),
            n_specaug=getattr(CFG, 'n_specaug', 2),
        ).to(device)
    else:
        # effv2s: No-resize transform
        print("No-resize transform")
        lms_transform = LogMelTransform(
            CFG.mel_spectrogram_params, CFG.top_db,
            filt_aug_db_range=CFG.filt_aug_db_range,
            filt_aug_n_band=CFG.filt_aug_n_band,
            filt_aug_min_bw=CFG.filt_aug_min_bw,
        ).to(device)
    
    # --- SpecAugment ---
    spec_aug_fn = None
    spec_aug_torch = None
    if hasattr(CFG, 'aug_flip_p') and hasattr(CFG, 'aug_dropout_p'):
        # Albumentations-based specaug (effv2s)
        spec_aug_fn = build_spec_aug(CFG)
    elif CFG.is_hgnet:
        # Torchaudio SpecAugment (HGNet)
        spec_aug_torch = SpecAugment().to(device)
        
    # --- MixUp ---
    # mixup = MixUp(alpha=getattr(CFG, 'mixup_alpha', 1.0),
    #                theta=getattr(CFG, 'mixup_theta', 0.8))
    
    # --- Model ---
    model = BirdModel(
        model_name=CFG.model_name,
        pretrained=CFG.pretrained,
        # pretrain_ckpt=pretrain_ckpt,
        drop_path_rate=CFG.drop_path_rate,
        drop_rate=CFG.drop_rate,
        num_classes=N_CLASSES,
        head_dropout=CFG.head_dropout,
        n_mels=CFG.mel_spectrogram_params["n_mels"],
        use_hgnet=CFG.is_hgnet,
        use_effb3=CFG.use_effb3,
        use_distill=CFG.use_distill,
        embed_dim=CFG.embed_dim, # getattr(CFG, 'perch_embed_dim', 1536),
        detach_cls_branch=CFG.detach_cls_branch, # getattr(CFG, 'detach_cls_branch', False),
    ).to(device)

    print(f"backbone_dim: {model.backbone_dim}, GeM p: {model.gem.p.item():.2f}")
    if CFG.is_hgnet:
        print(f"att_block aux_pool: '{model.att_block.aux_pool}'")

    bce = nn.BCEWithLogitsLoss()

    num_devices = torch.cuda.device_count()
    if CFG.use_dp and num_devices > 1:
        model = nn.DataParallel(model)

    # --- EMA ---
    ema = None
    if CFG.has_ema:
        ema = ModelEMA(model, decay=cfg.ema_decay)

    # --- Teacher (self-distillation) ---
    teacher = None
        
    # --- Perch ONNX teacher ---
    perch = None
    use_perch_distill = getattr(CFG, 'use_perch_distill', hasattr(CFG, 'perch_hf_repo'))
    if CFG.use_perch_distill:
        onnx_path = hf_hub_download(repo_id=CFG.perch_hf_repo, filename=CFG.perch_hf_file)
        
        perch_kwargs = dict(
            onnx_path=onnx_path,
            prefer_cuda=torch.cuda.is_available(),
            sr=CFG.sr, duration=CFG.duration,
            perch_input_sec=getattr(CFG, 'perch_input_sec', CFG.duration),
            perch_embed_dim=CFG.perch_embed_dim,
        )
        if CFG.has_ema:
            perch_kwargs['sub_batch_size'] = 16
        perch = PerchTeacher(**perch_kwargs)

    # --- Optimizer ---
    optimizer = torch.optim.AdamW(
        params=model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    # optimizer.zero_grad()

    # --- Scheduler ---
    if CFG.is_hgnet:
        warmup_steps = CFG.warmup_epoch * len(trn_loader)
        warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                          total_iters=warmup_steps)
        cosine = CosineAnnealingLR(optimizer,
                                   T_max=CFG.max_epoch * len(trn_loader) - warmup_steps,
                                   eta_min=8e-5)
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                                 milestones=[warmup_steps])
    else:
        scheduler = OneCycleLR(
            optimizer, max_lr=CFG.lr, epochs=CFG.max_epoch,
            steps_per_epoch=len(trn_loader),
            pct_start=CFG.warmup_epoch / CFG.max_epoch,
            div_factor=25, final_div_factor=4.0,
        )

    scaler = make_grad_scaler(device, enabled=CFG.use_amp)
    
    loss_func = nn.BCEWithLogitsLoss()
    # grad_scaler = torch.GradScaler(enabled=CFG.use_amp)

    stopper = EarlyStoppingAndTopKSaver(fold_id=fold_id, patience=5, top_k=3, out_dir=output_dir)

    result_list = []
    best_val_score = 0.0

    for epoch in trange(CFG.max_epoch):
        epoch_start = time()
        # trn_loss = 0.0
        # tmp_mixup = mixup if epoch >= CFG.warmup_epoch else None

        # --- MixUp Control (CFGからモードを取得) ---
        mixup_mode = getattr(CFG, 'mixup_mode', 'none')
        is_spec_mixup_active = mixup_mode in ['spectrogram', 'both']
        is_wave_mixup_active = mixup_mode in ['waveform', 'both']

        # Spectrogram MixUp の初期化
        if is_spec_mixup_active and epoch >= CFG.warmup_epoch:
            tmp_mixup = MixUp(
                alpha=getattr(CFG, 'spec_mixup_alpha', 1.0),
                theta=getattr(CFG, 'spec_mixup_theta', 0.8)
            )
        else:
            tmp_mixup = None  # Noneにすれば学習ループ内でスキップされる

        # Waveform MixUp のフラグ
        do_wave_mixup = is_wave_mixup_active and epoch >= CFG.warmup_epoch and  np.random.random() < CFG.wave_mixup_prob

        model = model.train()
        lms_transform.train()
        
        total_loss = total_cls = total_dist = total_clip = total_frame = 0.0
        n_batches = 0
        
        use_distill = perch is not None
        use_pseudo = teacher is not None
        
        for batch in trn_loader:
            batch = to_device(batch, device, non_blocking=True)
            wave, label = batch[0], batch[1]
            
            # lms = lms_transform_tr(wave)

            # Perch embedding (before GPU transfer for ONNX efficiency)
            if use_distill:
                with torch.no_grad():
                    perch_emb = perch.embed(wave).to(device, non_blocking=True)
            else:
                perch_emb = None

            # ターゲットEmbeddingを初期化 (MixUpでどんどんブレンドされていく)
            target_emb = perch_emb

            # 2. --- Waveform MixUp ---
            if do_wave_mixup:
                b = wave.shape[0]
                w_alpha = getattr(CFG, 'wave_mixup_alpha', 0.4)
                # Beta分布からサンプリング
                lam_w = torch.distributions.Beta(w_alpha, w_alpha).sample((b,)).to(device)
                lam_w = torch.maximum(lam_w, 1 - lam_w).float()
                idx_w = torch.randperm(b, device=device)

                # 波形レベルでのブレンド
                wave = lam_w[:, None] * wave + (1.0 - lam_w[:, None]) * wave[idx_w]
                # ラベルのブレンド (Hard Mixup)
                label = torch.maximum(label, label[idx_w])

                # 特徴量のブレンド (同じ比率で混ぜる)
                if use_distill:
                    target_emb = lam_w[:, None] * target_emb + (1.0 - lam_w[:, None]) * target_emb[idx_w]

            # 3. --- スペクトログラムへ変換 ---
            # 波形MixUpがかかっていれば、すでに混ざった音がLMSになる
            lms = lms_transform(wave)
            
            # --- MixUp ---
            if tmp_mixup is not None:
                lms, label, lam, idx = tmp_mixup(lms, label)
                # target_emb が存在する場合、さらにSpecMixUpの比率で重ねてブレンドする
                if use_distill and target_emb is not None:
                    target_emb = (
                        lam_s[:, None] * target_emb + (1.0 - lam_s[:, None]) * target_emb[idx_s]
                    )

            # --- SpecAugment (after MixUp) ---
            if spec_aug_fn is not None:
                lms = apply_spec_aug(lms, spec_aug_fn)
    
            if spec_aug_torch is not None:
                lms = spec_aug_torch(lms)

            # --- Forward ---
            with torch.autocast(
                device_type=device.type,
                enabled=(scaler.is_enabled() and device.type == "cuda")
            ):
                if use_distill:
                    clipwise, framewise, distill_emb = model(lms, return_distill=True)
                else:
                    clipwise, framewise = model(lms, return_distill=False)
                    distill_emb = None
                    
                # Classification loss
                clip_w, frame_w = CFG.loss_weights
                loss_clip = bce(clipwise, label)

                # Handle framewise: for HGNet aux_pool mode (framewise may be None)
                if framewise is not None:
                    if framewise.dim() == 2:
                        # HGNet aux_pool mode: framewise is already pooled to (B, n_classes)
                        loss_frame = bce(framewise, label)
                    else:
                        # Standard: (B, n_classes, T) -> max_pool over time
                        loss_frame = bce(framewise.max(dim=2)[0], label)
                    cls_loss  = clip_w * loss_clip + frame_w * loss_frame
                else:
                    loss_frame = torch.tensor(0.0, device=device)
                    cls_loss  = loss_clip
                # Distillation loss
                if use_distill:
                    dist_loss = compute_distill_loss(distill_emb, target_emb, loss_type=CFG.distill_loss_type)
                    loss = cls_loss + CFG.alpha_distill * dist_loss
                else:
                    dist_loss = torch.tensor(0.0, device=device)
                    loss = cls_loss
            
            # --- Backward ---
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            total_loss += loss.item()
            total_cls += cls_loss.item()
            total_dist += dist_loss.item()
            total_clip += loss_clip.item()
            total_frame += loss_frame.item()
            n_batches += 1
            
            # trn_loss += loss.item()
            del batch, wave, label, lms, clipwise, framewise # logits
    
        # trn_loss /= len(trn_loader)
        n = max(n_batches, 1)
        trn_loss = total_loss / n

        # --- Gradient clipping after scaler.step (HGNet seed2) ---
        if CFG.grad_clip_norm is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)

        # --- Validate ---
        lms_transform.eval()
        model = model.eval()
        logit_list  = []
        label_list = []
        for batch in val_loader:
            lms = lms_transform(to_device(batch[0], device, non_blocking=True))
            # lms = lms_transform_vl(to_device(batch["wave"], device, non_blocking=True))
            # with torch.no_grad() and torch.autocast(device.type, enabled=grad_scaler.is_enabled()):
            with torch.autocast(
                device_type=device.type,
                enabled=(scaler.is_enabled() and device.type == "cuda")
            ):
                # logits = model(lms.detach())
                clipwise, _ = model(lms)
            
            # logit_list.append(logits.detach().cpu())
            # logit_list.append(logits["lse_logits"].detach().cpu())
            logit_list.append(clipwise.detach().cpu())
            label_list.append(batch[1].detach())
            del batch, lms, clipwise # logits
        
        logits = torch.cat(logit_list, axis=0)
        labels = torch.cat(label_list, axis=0)

        # 1. テンソルの状態で各サンプルのLossを計算だけしておく
        loss_fn_none = torch.nn.BCEWithLogitsLoss(reduction='none')
        sample_losses = loss_fn_none(logits, labels).mean(dim=1).numpy()

        val_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels).item()
    
        logits = logits.numpy()
        labels = labels.numpy()
        mask = labels.sum(axis=0) > 0
        
        val_score = roc_auc_score(labels[:, mask], logits[:, mask], average="macro")

        if val_score > best_val_score:
            best_val_score = val_score
            if CFG.use_dp and num_devices > 1:
                best_state = {k: v.detach().cpu() for k, v in model.module.state_dict().items()}
            else:
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_val_pred  = logits

        epoch_end = time()
        result_list.append(
            [epoch, scheduler.get_last_lr()[0], trn_loss, val_loss, val_score, epoch_end - epoch_start])
        print(
            "[epoch {}] lr={:.6f}, trn_loss={:.5f}, val_loss={:.5f}, val_score={:.5f}. elapsed={:.2f}".format(
                *result_list[-1]
            ))

        # Stopperを呼び出す
        stopper(epoch, val_score, model.state_dict())

        if stopper.early_stop:
            print(f"Early stopping triggered at epoch {epoch}.")
            break # 学習ループを抜ける

    # --- Model Soups (学習ループ直後) ---
    final_model_path = os.path.join(output_dir, f"souped_best_model_fold{fold_id}.pt")
    average_top_k_checkpoints(stopper, final_model_path)

    # save training process 
    result_df = pd.DataFrame(
        result_list, columns=["epoch", "lr", "trn_loss", "val_loss", "val_score", "elapsed_time"])
    result_df.to_csv(output_dir / f"result_df_fold{fold_id}.csv", index=False)
    # save best model
    torch.save(best_state, output_dir / f"best_model_fold{fold_id}.pt")
    # save val_pred by best model
    np.save(output_dir / f"best_val_pred_fold{fold_id}.npy", best_val_pred)

In [46]:
# Build data
taxonomy = pd.read_csv(CFG.data_dir / "taxonomy.csv")
min_samples = getattr(CFG, 'min_samples_per_class', 0)
use_unlabeled = getattr(CFG, 'use_unlabeled_sc', False)
unl_chunks = getattr(CFG, 'unlabeled_chunks_per_file', 2)

print(f"min_samples: {min_samples}")
print(f"use_unlabeled: {use_unlabeled}")

df, classes, labels_arr = build_combined_df(
    CFG.data_dir, taxonomy,
    use_unlabeled_sc=use_unlabeled,
    unlabeled_chunks_per_file=unl_chunks,
    min_samples_per_class=min_samples,
)
print(f"Total: {len(df)} (focal={(df['source']=='focal').sum()}, "
      f"soundscape={(df['source']=='soundscape').sum()})")

df = assign_folds(df, labels_arr, CFG.n_folds, CFG.seed)
for f in tqdm(range(CFG.n_folds)):
    ft = ((df["source"] == "focal") & (df["fold"] != f)).sum()
    fv = ((df["source"] == "focal") & (df["fold"] == f)).sum()
    sc = (df["source"] == "soundscape").sum()
    print(f"  fold {f}: train={ft+sc} (focal={ft}, sc={sc}), val={fv}")

min_samples: 66
use_unlabeled: False
Rating filter (2.5): 35549 → 34319 (removed 1230)
[Execute Upsampling]
[Upsample] 64 rare species: 35058 -> 38793 samples (+3735 copies)
Total: 38793 (focal=38054, soundscape=739)


0it [00:00, ?it/s]

0it [00:00, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  fold 0: train=29267 (focal=28528, sc=739), val=9526
  fold 1: train=29299 (focal=28560, sc=739), val=9494
  fold 2: train=29262 (focal=28523, sc=739), val=9531
  fold 3: train=29290 (focal=28551, sc=739), val=9503


In [47]:
df.head()

,audio_id,filename,primary_label,source,start_sec,end_sec,label_list,fold
0,1161364/iNat1216197,1161364/iNat1216197.ogg,1161364,focal,0,0,[1161364],0
1,1161364/iNat1114648,1161364/iNat1114648.ogg,1161364,focal,0,0,[1161364],1
2,1161364/iNat810195,1161364/iNat810195.ogg,1161364,focal,0,0,[1161364],0
3,1161364/iNat818781,1161364/iNat818781.ogg,1161364,focal,0,0,[1161364],1
4,1161364/iNat556514,1161364/iNat556514.ogg,1161364,focal,0,0,[1161364],3


In [48]:
print(list(filter(lambda x: x[0][:2] != "__", CFG.__dict__.items())))

dry_run = False # True False

for fold_id in CFG.FOLDS:
    train_one_fold(df, fold_id, device, dry_run=dry_run)
    if dry_run:
        break

[('FOLDS', [0]), ('max_epoch', 35), ('warmup_epoch', 5), ('batch_size', 64), ('lr', 0.0008), ('weight_decay', 0.0006), ('model_name', 'hgnetv2_b0.ssld_stage2_ft_in1k'), ('pretrained', True), ('drop_path_rate', 0.15), ('drop_rate', 0.2), ('head_dropout', 0.45), ('lse_temperature', 1.0), ('is_hgnet', True), ('use_effb3', True), ('has_ema', False), ('use_distill', True), ('detach_cls_branch', False), ('embed_dim', 1536), ('sampling_gamma', -0.5), ('upsample_min_count', 100), ('use_weighted_sampler', False), ('context_sec', 5), ('target_sec', 5), ('segment_sec', 5), ('mel_spectrogram_params', {'sample_rate': 32000, 'n_fft': 2048, 'win_length': 1024, 'hop_length': 512, 'f_min': 20, 'f_max': 16000, 'n_mels': 128, 'power': 2.0, 'center': True, 'pad_mode': 'reflect', 'norm': 'slaney', 'mel_scale': 'htk'}), ('lms_shape', (128, 313)), ('top_db', 80.0), ('mixup', {'alpha': 1.0, 'theta': 0.8}), ('use_amp', True), ('use_dp', False), ('outer_weight', 1.0), ('smooth_power', 1.0), ('FREQ_MASK_PARAM', 

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


model.safetensors:   0%|          | 0.00/24.1M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/random.py:187: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


backbone_dim: 2048, GeM p: 3.00
att_block aux_pool: 'topk'


perch_v2_no_dft.onnx:   0%|          | 0.00/413M [00:00<?, ?B/s]

[Perch] providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] embed_idx=0


  0%|          | 0/35 [00:00<?, ?it/s]

[epoch 0] lr=0.000224, trn_loss=0.12870, val_loss=0.03087, val_score=0.58077. elapsed=564.26
[epoch 1] lr=0.000368, trn_loss=0.09748, val_loss=0.02598, val_score=0.83553. elapsed=551.47
[epoch 2] lr=0.000512, trn_loss=0.08043, val_loss=0.01965, val_score=0.91529. elapsed=546.47
[epoch 3] lr=0.000656, trn_loss=0.06695, val_loss=0.01662, val_score=0.93748. elapsed=551.15
[epoch 4] lr=0.000800, trn_loss=0.05885, val_loss=0.01571, val_score=0.94480. elapsed=555.36
[epoch 5] lr=0.000798, trn_loss=0.05395, val_loss=0.01499, val_score=0.94758. elapsed=554.53
[epoch 6] lr=0.000792, trn_loss=0.05021, val_loss=0.01433, val_score=0.95123. elapsed=551.16
[epoch 7] lr=0.000782, trn_loss=0.04764, val_loss=0.01382, val_score=0.95156. elapsed=549.59
[epoch 8] lr=0.000769, trn_loss=0.07935, val_loss=0.01951, val_score=0.95179. elapsed=559.43
[epoch 9] lr=0.000752, trn_loss=0.04465, val_loss=0.01398, val_score=0.95745. elapsed=551.65
[epoch 10] lr=0.000731, trn_loss=0.04297, val_loss=0.01296, val_score=

## Calculate OOF Score

In [49]:
# def rank_normalize(x):
#     r_x = np.zeros_like(x)
    
#     for i in range(x.shape[1]):
#         r_x_i = pd.Series(x[:, i]).rank(method="max")
#         r_x[:, i] = r_x_i / r_x_i.shape[0]

#     return r_x

### calculate CV score of predictions by torch model (on GPU)

In [50]:
# oof_pred_trn = np.zeros((len(train_df), N_CLASSES))
# oof_pred_trn_rank = np.zeros((len(train_df), N_CLASSES))

# for fold_id, (trn_idxs, val_idxs) in enumerate(train_val_splits):
#     val_pred = np.load(f"best_val_pred_fold{fold_id}.npy")
    
#     # ★追加: dry_run等で予測件数が少ない場合、インデックスもそれに合わせて切り捨てる
#     if len(val_pred) < len(val_idxs):
#         val_idxs = val_idxs[:len(val_pred)]
        
#     oof_pred_trn[val_idxs] = val_pred
#     oof_pred_trn_rank[val_idxs] = rank_normalize(val_pred)
    
#     if dry_run:
#         break

In [51]:
# print("auc for raw pred :", roc_auc_score(labels_arr, oof_pred_trn))
# print("auc for rank pred:", roc_auc_score(labels_arr, oof_pred_trn_rank))

## convert torch models to openvino models and save them

In [52]:
# # 2. エクスポート関数の修正
# def convert_torch_to_onnx_to_ov(
#     cfg,
#     model_path: Path,
#     out_dir: Path,
# ):
#     """convert torch model to openvino model via onnx model"""
#     # torch_model = LSEModel(
#     #     cfg.model_name,
#     #     pretrained=False,
#     #     drop_path_rate=cfg.drop_path_rate,
#     #     num_classes=N_CLASSES,
#     #     head_dropout=cfg.head_dropout,
#     #     # lse_temperature=cfg.lse_temperature,
#     #     input_shape=cfg.lms_shape,
#     #     # context_sec=cfg.context_sec,
#     #     # target_sec=cfg.target_sec,
#     #     # outer_weight=cfg.outer_weight,
#     #     smooth_power=cfg.smooth_power,
#     # )
#     torch_model = BirdModel(
#         model_name=cfg.model_name,
#         pretrained=False,
#         # pretrain_ckpt=pretrain_ckpt,
#         drop_path_rate=cfg.drop_path_rate,
#         drop_rate=cfg.drop_rate,
#         num_classes=N_CLASSES,
#         head_dropout=cfg.head_dropout,
#         n_mels=cfg.mel_spectrogram_params["n_mels"],
#         use_hgnet=cfg.is_hgnet,
#         use_effb3=cfg.has_ema,
#         use_distill=cfg.use_distill,
#         embed_dim=cfg.embed_dim, # getattr(CFG, 'perch_embed_dim', 1536),
#         detach_cls_branch=cfg.detach_cls_branch, # getattr(CFG, 'detach_cls_branch', False),
#     )
#     torch_model.load_state_dict(torch.load(model_path))
#     torch_model = torch_model.eval()

#     # OOF検証用モデル (10秒用: 513フレーム)
#     dummy_input = torch.randn(1, 1, 256, 513, dtype=torch.float32)

#     onnx_path = out_dir / f"oof_{model_path.stem}.onnx"
#     _ = torch.onnx.export(
#         torch_model,  # ★ FCNInferenceWrapperではなく、元のモデルを渡す
#         (dummy_input,),
#         onnx_path,
#         opset_version=11,
#         do_constant_folding=True,
#         input_names=["input"],
#         output_names=["output"],
#         # ★ inputの時間軸(3)を動的に指定することで、512でも513でもエラーにならなくなります
#         dynamic_axes={"input": {0: "batch_size", 3: "time"}, "output": {0: "batch_size"}},
#     )

#     ov_model = ov.convert_model(onnx_path)
#     onnx_path.unlink()
#     ov.save_model(ov_model, out_dir / f"oof_{model_path.stem}.xml")

#     return ov_model

In [53]:
# lms_transform = LogMelSpectrogramTransform(
#     CFG.mel_spectrogram_params, top_db=CFG.top_db, # lms_shape=CFG.lms_shape
# ).eval()

# joblib.dump(lms_transform, "log_mel_spectrogram.jb")

In [54]:
# out_dir = Path.cwd()
# for fold_id in CFG.FOLDS:
#     model_state_path = out_dir / f"best_model_fold{fold_id}.pt"
#     ov_model = convert_torch_to_onnx_to_ov(CFG, model_state_path, out_dir)

In [55]:
# def init_layer(layer):
#     nn.init.xavier_uniform_(layer.weight)
#     if hasattr(layer, "bias") and layer.bias is not None:
#         layer.bias.data.fill_(0.)

# def init_bn(bn):
#     bn.bias.data.fill_(0.)
#     bn.weight.data.fill_(1.0)

# class GeM1d(nn.Module):
#     def __init__(self, p=3.0, kernel_size=3, eps=1e-6):
#         super().__init__()
#         self.p = nn.Parameter(torch.ones(1) * p)
#         self.kernel_size = kernel_size
#         self.eps = eps

#     def forward(self, x):
#         x = x.clamp(min=self.eps)
#         return F.avg_pool1d(
#             x.pow(self.p), kernel_size=self.kernel_size,
#             stride=1, padding=self.kernel_size // 2,
#         ).pow(1.0 / self.p)

# class AttBlock(nn.Module):
#     def __init__(self, in_feat, out_feat):
#         super().__init__()
#         self.att = nn.Conv1d(in_feat, out_feat, 1, bias=True)
#         self.cla = nn.Conv1d(in_feat, out_feat, 1, bias=True)
#         init_layer(self.att)
#         init_layer(self.cla)

#     def forward(self, x):
#         norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
#         framewise = self.cla(x)
#         clipwise = torch.sum(norm_att * framewise, dim=2)
#         return clipwise

# class DistillHead(nn.Module):
#     def __init__(self, backbone_dim, embed_dim=1536):
#         super().__init__()
#         self.proj = nn.Linear(backbone_dim, embed_dim)
#         init_layer(self.proj)

#     def forward(self, feat_map):
#         return self.proj(feat_map.mean(dim=[2, 3]))

# class BirdModel(nn.Module):
#     def __init__(self, model_name, pretrained=False, drop_path_rate=0.15, drop_rate=0.2,
#                  num_classes=234, head_dropout=0.35, n_mels=128):
#         super().__init__()
#         self.bn0 = nn.BatchNorm2d(n_mels)
#         init_bn(self.bn0)

#         self.backbone = timm.create_model(
#             model_name, pretrained=pretrained, in_chans=1,
#             global_pool="", num_classes=0,
#             drop_path_rate=drop_path_rate, drop_rate=drop_rate,
#         )
#         with torch.no_grad():
#             n_feat = self.backbone(torch.randn(1, 1, n_mels, n_mels)).shape[1]

#         self.gem = GeM1d(p=3.0, kernel_size=3)
#         self.fc1 = nn.Linear(n_feat, n_feat, bias=True)
#         self.att_block = AttBlock(n_feat, num_classes)
#         self.dropout = nn.Dropout(head_dropout)
#         self.distill_head = DistillHead(n_feat, 1536)

#         init_layer(self.fc1)

#     def forward(self, x):
#         x = x.transpose(1, 2) 
#         x = self.bn0(x)
#         x = x.transpose(1, 2) 

#         feat = self.backbone(x)
#         feat = feat.mean(dim=2)
        
#         feat = self.gem(feat)
#         feat = self.dropout(feat)
#         feat = feat.transpose(1, 2)
#         feat = nn.functional.relu(self.fc1(feat))
#         feat = feat.transpose(1, 2)
#         feat = self.dropout(feat)

#         clipwise = self.att_block(feat)
#         return clipwise

# class LogMelSpectrogramTransform(nn.Module):
#     def __init__(self, mel_params: dict, top_db: float):
#         super().__init__()
#         self.mel_transform = torchaudio.transforms.MelSpectrogram(**mel_params)
#         self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=top_db)
#         self.top_db = top_db

#     @torch.no_grad()
#     def forward(self, wave: torch.Tensor) -> torch.Tensor:
#         mel = self.mel_transform(wave)
#         lms = self.db(mel)
        
#         lms = torch.clamp((lms + self.top_db) / self.top_db, 0.0, 1.0)
#         return lms[:, None, :, :]
        
# print('Done')

In [56]:
# # 1. ONNXエクスポート用のラッパークラス（変更なし）
# class InferenceWrapper(nn.Module):
#         def __init__(self, base_model):
#             super().__init__()
#             self.base_model = base_model
            
#         def forward(self, x):
#             clipwise = self.base_model(x)
#             blended_logits = clipwise
            
#             return blended_logits

# # 2. エクスポート関数の修正（本番推論用）
# def convert_torch_to_onnx_to_ov(
#     cfg,
#     model_path: Path,
#     out_dir: Path,
# ):
#     """convert torch model to openvino model via onnx model"""
#     # torch_model = LSEModel(
#     #     cfg.model_name,
#     #     pretrained=False,
#     #     drop_path_rate=cfg.drop_path_rate,
#     #     num_classes=N_CLASSES,
#     #     head_dropout=cfg.head_dropout,
#     #     # lse_temperature=cfg.lse_temperature,
#     #     input_shape=cfg.lms_shape,
#     #     # context_sec=cfg.context_sec,
#     #     # target_sec=cfg.target_sec,
#     #     # outer_weight=cfg.outer_weight,
#     #     smooth_power=cfg.smooth_power,
#     # )
#     torch_model = BirdModel(
#         model_name=cfg.model_name,
#         pretrained=False,
#         # pretrain_ckpt=pretrain_ckpt,
#         drop_path_rate=cfg.drop_path_rate,
#         drop_rate=cfg.drop_rate,
#         num_classes=N_CLASSES,
#         head_dropout=cfg.head_dropout,
#         n_mels=cfg.mel_spectrogram_params["n_mels"],
#     )
#     torch_model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=False)
#     torch_model = torch_model.eval()

#     # ★ 変更点1: モデルをラッパーで包む
#     wrapped_model = InferenceWrapper(torch_model).eval()

#     # ★ 変更点2: 本番推論用モデル (60秒一括処理用: 3073フレーム)
#     dummy_input = torch.randn(1, 1, 256, 3073, dtype=torch.float32)

#     onnx_path = out_dir / f"{model_path.stem}.onnx"
#     _ = torch.onnx.export(
#         wrapped_model,  # ★ 変更点3: torch_model ではなく wrapped_model を渡す
#         (dummy_input,),
#         str(onnx_path),
#         opset_version=18,
#         do_constant_folding=True,
#         input_names=["input"],
#         output_names=["output"],
#         # ★ 変更点4: 推論時は常に60秒固定で切り出すため、時間軸(3)の動的指定を外し、バッチサイズのみ可変にする
#         dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
#     )

#     ov_model = ov.convert_model(onnx_path)
#     onnx_path.unlink()
    
#     # ★ 保存名も本番用とわかるように変更
#     save_path = out_dir / f"{model_path.stem}.xml"
#     ov.save_model(ov_model, out_dir / f"{model_path.stem}.xml")
#     print(f"Exported Submission Model: {save_path}")

#     return ov_model

In [57]:
# out_dir = Path.cwd()
# for fold_id in CFG.FOLDS:
#     model_state_path = out_dir / f"best_model_fold{fold_id}.pt"
#     ov_model = convert_torch_to_onnx_to_ov(CFG, model_state_path, out_dir)

In [58]:
# # 1. ONNXエクスポート用のラッパークラス（変更なし）
# class InferenceWrapper(nn.Module):
#         def __init__(self, base_model):
#             super().__init__()
#             self.base_model = base_model
            
#         def forward(self, x):
#             clipwise = self.base_model(x)
#             blended_logits = clipwise
            
#             return blended_logits

# # 2. エクスポート関数の修正（本番推論用）
# def convert_torch_to_onnx_to_ov(
#     cfg,
#     model_path: Path,
#     out_dir: Path,
# ):
#     """convert torch model to openvino model via onnx model"""
#     # torch_model = LSEModel(
#     #     cfg.model_name,
#     #     pretrained=False,
#     #     drop_path_rate=cfg.drop_path_rate,
#     #     num_classes=N_CLASSES,
#     #     head_dropout=cfg.head_dropout,
#     #     # lse_temperature=cfg.lse_temperature,
#     #     input_shape=cfg.lms_shape,
#     #     # context_sec=cfg.context_sec,
#     #     # target_sec=cfg.target_sec,
#     #     # outer_weight=cfg.outer_weight,
#     #     smooth_power=cfg.smooth_power,
#     # )
#     torch_model = BirdModel(
#         model_name=cfg.model_name,
#         pretrained=False,
#         # pretrain_ckpt=pretrain_ckpt,
#         drop_path_rate=cfg.drop_path_rate,
#         drop_rate=cfg.drop_rate,
#         num_classes=N_CLASSES,
#         head_dropout=cfg.head_dropout,
#         n_mels=cfg.mel_spectrogram_params["n_mels"],
#     )
#     torch_model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=False)
#     torch_model = torch_model.eval()

#     # ★ 変更点1: モデルをラッパーで包む
#     wrapped_model = InferenceWrapper(torch_model).eval()

#     # ★ 変更点2: 本番推論用モデル (60秒一括処理用: 3073フレーム)
#     dummy_input = torch.randn(1, 1, 256, 3073, dtype=torch.float32)

#     onnx_path = out_dir / f"{model_path.stem}.onnx"
#     _ = torch.onnx.export(
#         wrapped_model,  # ★ 変更点3: torch_model ではなく wrapped_model を渡す
#         (dummy_input,),
#         str(onnx_path),
#         opset_version=18,
#         do_constant_folding=True,
#         input_names=["input"],
#         output_names=["output"],
#         # ★ 変更点4: 推論時は常に60秒固定で切り出すため、時間軸(3)の動的指定を外し、バッチサイズのみ可変にする
#         dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
#     )

#     ov_model = ov.convert_model(onnx_path)
#     onnx_path.unlink()
    
#     # ★ 保存名も本番用とわかるように変更
#     save_path = out_dir / f"{model_path.stem}.xml"
#     ov.save_model(ov_model, out_dir / f"{model_path.stem}.xml")
#     print(f"Exported Submission Model: {save_path}")

#     return ov_model

In [59]:
# out_dir = Path.cwd()
# for fold_id in CFG.FOLDS:
#     model_state_path = out_dir / f"souped_best_model_fold{fold_id}.pt"
#     ov_model = convert_torch_to_onnx_to_ov(CFG, model_state_path, out_dir)

### calculate score of predictions by openvino model on CPU

In [60]:
# lms_transform = LogMelSpectrogramTransform(
#     CFG.mel_spectrogram_params, top_db=CFG.top_db, # lms_shape=CFG.lms_shape
# ).eval().to(device)

# ov_model_list = []
# for fold_id in CFG.FOLDS:
#     ov_model_path = f"oof_best_model_fold{fold_id}.xml"
#     compiled_model = ov.compile_model(
#         ov_model_path, "CPU", {
#             "PERFORMANCE_HINT": "THROUGHPUT",
#             "INFERENCE_NUM_THREADS": 4,
#             'NUM_STREAMS': 2,
#         }
#     )
#     ov_model_list.append(compiled_model)

# gc.collect()

## EOF